In [ ]:
import glob, os, json, re, io
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import scipy.stats as ss
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.cluster import KMeans, AgglomerativeClustering
from statannotations.Annotator import Annotator

sns.set_context('talk')

import importlib
import myutil
importlib.reload(myutil)
from myutil import *

import warnings
# Suppress warnings
warnings.filterwarnings('ignore')

In [ ]:
# sns.set_context("talk")
# fig, ax = plt.subplots(1, 1, figsize=(4.5, 5.5))

# # make sure model order is fixed
# all_params_df["model_name"] = pd.Categorical(
#     all_params_df["model_name"],
#     categories=m2_list,
#     ordered=True
# )

# # connect each subject across models
# for sub, g in all_params_df.sort_values("model_name").groupby("subID"):
#     ax.plot(
#         g["model_name"],
#         g["bic"],
#         color="gray",
#         alpha=0.25,
#         linewidth=0.8,
#         zorder=1
#     )

# # raw subject-level points
# sns.swarmplot(
#     data=all_params_df,
#     x="model_name",
#     y="bic",
#     order=m2_list,
#     color="black",
#     size=2.5,
#     alpha=0.25,
#     ax=ax,
#     zorder=2
# )

# # optional mean bar / point overlay
# sns.pointplot(
#     data=all_params_df,
#     x="model_name",
#     y="bic",
#     order=m2_list,
#     color="gold",
#     errorbar="se",
#     markers="D",
#     linestyle="none",
#     ax=ax,
#     zorder=3
# )

# ax.set_xticklabels(model_names, rotation=45, ha="right")
# ax.set_xlabel("")
# ax.set_ylabel("BIC", fontsize=17)


# plt.tight_layout()

# if save:
#     plt.savefig(
#         f"../paper_figs/{folder}/bic_swarm_connected_{folder}.png",
#         bbox_inches="tight",
#         dpi=200
#     )

In [ ]:
# define colors here
outcome_palette = {'harvest':'#daa520', 'captured':"#bf00ff"}
outcome_colors = ['#daa520', "#bf00ff"] #win, lose
compromise_palette = {'high':'firebrick', 'low':'steelblue'}
compromise_colors = ['firebrick', 'steelblue'] #high, low
risk_palette = {'risk-averse': 'teal', 'risk-prone': 'salmon'}
risk_colors = ['teal', 'salmon'] #high, low

# Read Data

In [ ]:
## determine which batch to plot
# folder = 'expl'
folder = 'conf'

## define where to save figures
# save = True
save = False

In [ ]:
# # # #if expl
if folder == 'expl':
    df_group = pd.read_csv('../processed_data/parsed_group.csv', index_col=[0])
    df_idv = pd.read_csv('../processed_data/parsed_idv.csv', index_col=[0])
    print(f"{len(df_group['room'].unique())} rooms in total with {len(df_group['sub'].unique())} subjects")
else:
    df_group = pd.read_csv('../../pilot7_rep/processed_data/parsed_all_group.csv', index_col=[0])
    df_group['player_partner_diff'] = df_group['playerStep'] - df_group['partnerStep']
    df_idv_all = pd.read_csv('../../pilot7_rep/processed_data/parsed_all_idv.csv', index_col=[0])
    # df_idv_all = df_idv_all.drop('attack.1' , axis=1)
    df_idv_all['subID'] = df_idv_all['subID'] + 400
    df_idv = df_idv_all.query('trial <=120')
    df_idv2 = df_idv_all.query('trial >120')

    # print(f"{len(df_group['room'].unique())} rooms in total")
    # print(f"{len(df_idv['sub'].unique())} subjects in total")
    df_group['room'] = df_group['room'] + 200
    df_group['subID'] = df_group['subID'] + 400
    df_group['jointMoney'] = df_group.apply(lambda x: -20 if x['attack'] else x['jointMoney'], axis=1)

    df_group = pd.concat([df_group, 
                      pd.read_csv('../processed_data/parsed_group_conf.csv', index_col=[0])])
    # df_group = df_group.query('room !=52 and room != 72')
    df_idv = pd.concat([df_idv, 
                        pd.read_csv('../processed_data/parsed_idv_conf.csv', index_col=[0])])
    # df_idv = df_idv.query('sub != "665a00"')
    # qs = pd.read_csv('../processed_data/parsed_questionnaire_all.csv')
    print(f"{len(df_group['room'].unique())} rooms in total")
    print(f"{len(df_idv['sub'].unique())} subjects in total")
    
df_group = df_group[['block', 'trial', 'room', 'subID', 'attack', 'selfBlame', 'jointMoney', 'partnerStep', 'playerStep', 'prediction', 'step_rt', 'predatorType', 'player_partner_diff', 'finalStep', 'playerID', 'confidence']]
# df_perf_conf = pd.read_csv('../processed_data/group_perf_conf.csv')
num_subs = len(df_idv['subID'].unique())


In [ ]:
# c = pd.merge(df_group.query('predatorType==1').groupby(['room'], as_index=False)['playerStep'].mean(), 
#          df_group.query('predatorType==0').groupby(['room'], as_index=False)['playerStep'].mean(),
#          on='room')
# c.query('playerStep_x > playerStep_y')

In [ ]:
#add partner blame
df_group['partnerBlame'] = df_group.groupby(['room', 'trial', 'predatorType'])['selfBlame'].shift()
g = df_group.groupby(['room', 'trial', 'predatorType'])['selfBlame']
df_group['partnerBlame'] = g.shift(1).fillna(g.shift(-1))
df_group['pred_error'] = np.abs(df_group['prediction'] - df_group['partnerStep'])
#add paredator and num_encounter
df_idv['predator'] = df_idv['predatorType'].apply(lambda x: 'low-threat' if x==0 else 'high-threat')
df_idv['num_encounter'] =  df_idv.groupby(['subID', 'predatorType'])['trial'].cumcount() + 1
df_group['predator'] = df_group['predatorType'].apply(lambda x: 'low-threat' if x==0 else 'high-threat')
df_group['outcome'] = df_group['attack'].apply(lambda x: 'harvest' if x==False else 'captured')

In [ ]:
c = df_group.groupby(['subID'])['trial'].count()
c[c>60]

In [ ]:
# c = df_group.loc[df_group.step_rt==8].groupby(['sub', 'room'])['trial'].count()
# c[c>=20]
#are there people who played twice?
c = df_group.groupby(['subID'])['room'].apply(lambda x: len(np.unique(x)))
subs = c[c>1].index
#you may want to remove room 52/58, room 72/78
df_group.query('subID in @subs')[['subID', 'room']].drop_duplicates()

In [ ]:
# subs = df_group['sub'].unique()
# id_mappings = dict(zip(subs, np.arange(len(subs))))
# df_group['sub'] = df_group['sub'].apply(lambda x: id_mappings[x])
# df_idv = df_idv.query('sub in @subs')
# df_idv['sub'] = df_idv['sub'].apply(lambda x: id_mappings[x])
# print(f"{len(df_group['room'].unique())} rooms in total")


In [ ]:
import myutil
import importlib
importlib.reload(myutil)
from myutil import *



# save regression results to a table in img format
def save_reg_to_table(change_model, num_betas, change_formula, title, folder=folder):
    rdf = results_to_df(change_model, num_betas)
    rdf = format_df_for_print(rdf,)
    df_to_png(rdf, change_formula, title, folder)

# Fig1 Location Choice

In [ ]:
#is there a difference in location choice between different predators?
#in both group and individual condition
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# gg = df_group.groupby(['sub', 'room', 'predatorType'], as_index=False)[['finalStep', 'playerStep']].mean()
# gi = df_idv.loc[df_idv.trial>60].groupby(['sub', 'predatorType'], as_index=False)['choice'].mean()
# sns.barplot(data=gi, x='predatorType', y='choice', ax=axes[0])
# axes[0].set_ylabel('individual choice')
# sns.barplot(data=gg, x='predatorType', y='playerStep', ax=axes[1])
# #sns.swarmplot(data=gg, x='predatorType', y='stepChoice', ax=axes[1])
# axes[1].set_ylabel('group choice')
# axes[0].set_xticklabels(['safe', 'dangerous'])
# axes[1].set_xticklabels(['safe', 'dangerous'])


# Set the font for the plot
plt.rcParams['font.family'] = 'Arial'  


def get_predator_colors():
    return {0: "#009E73", 1: "#8E44AD"}  # Teal and Dark Purple

# Apply colors using the defined function
predator_colors = get_predator_colors()

# Create the figure and axes
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

# Data processing for individual and group foraging
gg = df_group.query('step_rt<8 and playerStep>0').groupby(['subID', 'room', 'predatorType'], 
                  as_index=False)[['finalStep', 'playerStep', 'jointMoney']].mean()
gi = df_idv.query('trial>60').groupby(['subID', 'predatorType'], as_index=False)[['choice', 'reward']].mean()
# gi = df_idv2.groupby(['sub', 'predatorType'], as_index=False)[['choice', 'reward']].mean()
print([len(gg), len(gi)])
# Plot for individual foraging
sns.kdeplot(data=gi, hue='predatorType', x='choice', ax=axes[0], fill=True, palette=predator_colors, 
             linewidth=3)
axes[0].set_title('Individual Foraging')
axes[0].set_xlabel('Location Choice')
axes[0].legend(['Dangerous', 'Safe'], title='Predator', fontsize=11, frameon=False, 
               facecolor='darkgrey', labelcolor='#4D4D4D', 
               )
##decrease font of legend title
axes[0].get_legend().get_title().set_fontsize('11')

# Plot for group foraging
sns.kdeplot(data=gg, hue='predatorType', x='playerStep', ax=axes[1], fill=True, palette=predator_colors, 
            linewidth=3)
axes[1].set_title('Dyad Foraging')
axes[1].set_xlabel('Location Choice')
axes[1].legend(['Dangerous', 'Safe'], title='Predator', fontsize=11, frameon=False, 
               facecolor='darkgrey', labelcolor='#4D4D4D', 
               )
##decrease font of legend title
axes[1].get_legend().get_title().set_fontsize('11')

# Tight layout for better spacing
plt.tight_layout()
plt.show()

#print mean and CI
def confidence_interval(data, confidence=0.95):
    data = np.array(data)
    mean = np.mean(data)
    n = len(data)
    stderr = ss.sem(data)  # Standard error of the mean
    margin_of_error = stderr * ss.t.ppf((1 + confidence) / 2., n - 1)
    return mean, mean - margin_of_error, mean + margin_of_error

# Example usage:
print(f"idv safe: m= {np.mean(gi.query('predatorType==0')['choice'])}, \
      se={ss.sem(gi.query('predatorType==0')['choice'])}")
print(f"idv dangerous: m= {np.mean(gi.query('predatorType==1')['choice'])}, \
      se={ss.sem(gi.query('predatorType==1')['choice'])}")
print(f"group safe: m= {np.mean(gg.query('predatorType==0')['playerStep'])}, \
      se={ss.sem(gg.query('predatorType==0')['playerStep'])}")
print(f"group dangerous: m= {np.mean(gg.query('predatorType==1')['playerStep'])}, \
      se={ss.sem(gg.query('predatorType==1')['playerStep'])}")
# mean, lower_bound, upper_bound = confidence_interval(data)
# print(f"Mean: {mean}, 95% Confidence Interval: ({lower_bound}, {upper_bound})")

##calculate variance 
print(f"idv safe: var= {np.var(gi.query('predatorType==0')['choice'])}")
print(f"idv dangerous: var= {np.var(gi.query('predatorType==1')['choice'])}")
print(f"group safe: var= {np.var(gg.query('predatorType==0')['playerStep'])}")
print(f"group dangerous: var= {np.var(gg.query('predatorType==1')['playerStep'])}")


##do t-test between individual and group foraging for each predator type

# Perform t-test for safe predator
ttest_safe = ss.ttest_ind(gi.query('predatorType==0')['choice'], gg.query('predatorType==0')['playerStep'])

# Perform t-test for dangerous predator
ttest_dangerous = ss.ttest_ind(gi.query('predatorType==1')['choice'], gg.query('predatorType==1')['playerStep'])

# Print the results
print(f"Safe Predator: t-value = {ttest_safe.statistic}, p-value = {ttest_safe.pvalue}")
print(f"Dangerous Predator: t-value = {ttest_dangerous.statistic}, p-value = {ttest_dangerous.pvalue}")




In [ ]:
def get_riskiness_wpair_by_predator_type(gi, gg):
    """
    Calculate riskiness with pair by predator type.
    takes in individual and group dataframes,
    """
    #merge individual and group data
    g = pd.merge(gi, gg, on=['subID', 'predatorType'])
    #rename a couple of columns
    g = g.rename({'choice':'individual', 'playerStep':'group'}, axis=1)
    g['predator']  = g['predatorType'].apply(lambda x: 'low-threat' if x==0 else 'high-threat')

    #add step / reward increase
    g['step_inc'] = g['group'] - g['individual']
    g['reward_inc'] = g['jointMoney'] / 2 - g['reward']

    #add risky / risk-averse
    group_stat = g.groupby(['room', 'predatorType'])['individual'].apply(lambda x: x.sort_values()).reset_index()
    group_stat['risky_wpair'] = ['risk-averse','risk-prone'] * (len(group_stat)//2)
    group_stat = pd.merge(g, group_stat).sort_values(by=['room', 'predatorType', 'individual'])

    return group_stat

In [ ]:
def filter_groups_by_riskiness(group_stat):
    """
    Filter out groups where individual risk is the same.
    """
    c = group_stat.groupby(['room', 'predatorType']).size()
    filtered_out = c[c==2].index
    print(f"keep {len(filtered_out)} (room, predator)")
    group_stat_filtered = group_stat[group_stat.set_index(['room', 'predatorType']).index.isin(filtered_out)]
    assert(len(group_stat_filtered)<=num_subs*2)
    
    return group_stat_filtered

In [ ]:
# get group stats
group_stat = get_riskiness_wpair_by_predator_type(gi, gg)


# #in cluster analyses, filter out groups where individual risk is the same
# c = group_stat.groupby(['room', 'predatorType']).size()
# filtered_out = c[c==2].index
# print(f"keep {len(filtered_out)} (room, predator)")
# group_stat_filtered = group_stat[group_stat.set_index(['room', 'predatorType']).index.isin(filtered_out)]
# assert(len(group_stat_filtered)<=num_subs*2)
# group_stat_filtered.head()

print(ss.ttest_rel(group_stat['individual'], group_stat['group']))


## ttest on idv vs group step for RA and RP
print("RA: dyad vs idv step:")
print(ss.ttest_rel(group_stat.query('risky_wpair=="risk-averse"')['group'], group_stat.query('risky_wpair=="risk-averse"')['individual']))
print("RP: dyad vs idv step:")
print(ss.ttest_rel(group_stat.query('risky_wpair=="risk-prone"')['group'], group_stat.query('risky_wpair=="risk-prone"')['individual']))
# group_stat['diff'] = group_stat['individual'] - group_stat['group']
# # print(ss.ttest_ind(group_stat['diff'], 0))


# ##ttest of diff between group and idv for each predator type
# print(ss.ttest_ind(group_stat.query('predatorType==0')['diff'], 0))
# print(ss.ttest_ind(group_stat.query('predatorType==1')['diff'], 0))



In [ ]:

def visualize_raw_data(g):
    fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)  
    plt.subplots_adjust(hspace=0.4)

    x_map = {'individual': 0, 'group': 1}

    for i, pred_type in enumerate([0, 1]):
        ax = axes[i]
        
        # Subset and sort
        data = g.loc[g.predatorType == pred_type].sort_values(by=['room', 'risky_wpair'])
        
        # Melt data
        melted_data = data[['subID', 'risky_wpair', 'individual', 'group']].melt(
            id_vars=['subID', 'risky_wpair'], 
            value_vars=['individual', 'group'], 
            var_name='variable', 
            value_name='value'
        )
        
        # Map variable to numeric + jitter
        noise = 0.05
        melted_data['y_jittered'] = melted_data['variable'].map(x_map) + \
                                    np.random.normal(0, noise, size=melted_data.shape[0])

        for phase, y_value in x_map.items():
            for order in risk_palette.keys():
                subset = melted_data[(melted_data['variable'] == phase) & (melted_data['risky_wpair'] == order)]
                
                kde = sns.kdeplot(subset['value'], linewidth=0, common_norm=False, color='gray', ax=ax)

                x_vals = kde.lines[-1].get_xdata()
                y_vals = kde.lines[-1].get_ydata()
                offset = noise * 2

                if phase == "individual":
                    ax.fill_betweenx(y=y_value - y_vals * 0.7 - offset,
                                     x1=x_vals, x2=y_value - offset,
                                     alpha=0.1, color=risk_palette[order])
                elif phase == "group":
                    ax.fill_betweenx(y=y_value + y_vals * 0.7 + offset,
                                     x1=x_vals, x2=y_value + offset,
                                     alpha=0.1, color=risk_palette[order])

        # Line plot
        sns.lineplot(data=melted_data, y='y_jittered', x='value', hue='risky_wpair',
                     units='subID', estimator=None, ax=ax, 
                     palette=risk_palette, linewidth=0.2, alpha=0.4, legend=False)

        # Scatter plot
        sns.scatterplot(data=melted_data, y='y_jittered', x='value', hue='risky_wpair',
                        palette=risk_palette, s=20, ax=ax, legend=False, alpha=0.4)

        # Pointplot for means
        sns.pointplot(data=melted_data, y='variable', x='value', hue='risky_wpair',
                      palette=risk_palette, scale=1, ax=ax, ci=95, dodge=0.1, legend=False)

        # Set yticks as phase labels
        ax.set_yticks([0, 1])
        ax.set_yticklabels(['Individual', 'Dyad'], rotation=90, va='center', fontsize=19)

        # Axis labels
        ax.set_ylabel('')
        ax.set_xlabel('Foraging Step', labelpad=15, fontsize=19)

        # Axis limits
        ax.set_xlim(0.5, 9.5)

    # Titles
    axes[0].set_title('Low-Threat Predator', fontsize=20)
    # axes[0].set_facecolor('orange')
    # axes[0].patch.set_alpha(0.05)
    axes[1].set_title('High-Threat Predator', fontsize=20)
    # axes[1].set_facecolor('blue')
    # axes[1].patch.set_alpha(0.05)

    # Custom legend
    legend_handles = [
        Line2D([0], [0], color='w', markerfacecolor=risk_palette['risk-averse'], 
               markersize=10, marker='o', label='risk-averse'),
        Line2D([0], [0], color='w', markerfacecolor=risk_palette['risk-prone'], 
               markersize=10, marker='o', label='risk-prone')
    ]
    fig.legend(handles=legend_handles, title='Riskiness within Dyad', ncol=2, 
               bbox_to_anchor=(0.9, 0.074), frameon=True, fontsize=13, title_fontsize=13)

    # plt.tight_layout()
    if save:
        plt.savefig(f'../figs/{folder}/behavior_{folder}.png', 
                bbox_inches='tight', dpi=200)


In [ ]:
visualize_raw_data(group_stat)

In [ ]:
#compare money increase between risky and risk-averse individuals
group_stat_filtered = filter_groups_by_riskiness(group_stat)
sns.boxplot(data=group_stat_filtered, y='step_inc', x='predatorType', hue='risky_wpair', palette=risk_palette)

#run a anova - no obvious difference between predator type and riskiness
group_stat_filtered['abs_step_inc'] = np.abs(group_stat_filtered['step_inc'])
model = smf.ols('abs_step_inc ~ C(predatorType) + C(risky_wpair)', data=group_stat_filtered)
anova_table = sm.stats.anova_lm(model.fit(), typ=2)
print(anova_table)

#then t-test
print(ss.ttest_ind(group_stat_filtered.query('risky_wpair=="risk-averse"')['abs_step_inc'], 
                   group_stat_filtered.query('risky_wpair=="risk-prone"')['abs_step_inc'],))
# print(ss.ttest_1samp(group_stat_filtered.query('risky_wpair=="risk-averse"')['step_inc'], 0))
# sample =group_stat_filtered.query('risky_wpair=="risk-prone"')['step_inc']
# mean_diff = 0 - np.mean(sample)  # or just np.mean(sample)
# std_dev = np.std(sample, ddof=1)  # use ddof=1 for sample std
# cohens_d = mean_diff / std_dev
# print(f"Cohen's d for risk-averse: {cohens_d}")

# print(ss.ttest_1samp(group_stat_filtered.query('risky_wpair=="risk-prone"')['step_inc'], 0))
# sample = group_stat_filtered.query('risky_wpair=="risk-prone"')['step_inc']
# mean_diff = 0 - np.mean(sample)  # or just np.mean(sample)
# std_dev = np.std(sample, ddof=1)  # use ddof=1 for sample std
# cohens_d = mean_diff / std_dev
# print(f"Cohen's d for risk-prone: {cohens_d}")

In [ ]:
# group_stat = pd.merge(group_stat, df_group.query('selfBlame>0').groupby(['subID'], as_index=False)['selfBlame'].mean())
# model = smf.ols('selfBlame ~ C(predatorType) + C(risky_wpair)', data=group_stat)
# anova_table = sm.stats.anova_lm(model.fit(), typ=2)
# print(anova_table)


# ego_bias = df_group.query('selfBlame>0').groupby(['subID', 'attack', 'predatorType'], as_index=False)['selfBlame'].mean()
# ego_bias = pd.merge(ego_bias.query('attack==True'), ego_bias.query('attack==False'), on=['subID', 'predatorType'], suffixes=['_lose', '_win'])
# ego_bias['ego_bias'] = ego_bias['selfBlame_win'] - ego_bias['selfBlame_lose']
# ego_bias = pd.merge(ego_bias, group_stat)
# model = smf.ols('ego_bias ~ C(predatorType) + C(risky_wpair)', data=ego_bias)
model = smf.ols('selfBlame ~ C(predatorType) + C(attack)', data=ego_bias)
anova_table = sm.stats.anova_lm(model.fit(), typ=2)
print(anova_table)

In [ ]:
# #compare money increase between risky and risk-averse individuals
group_stat_filtered = filter_groups_by_riskiness(group_stat)
# sns.boxplot(data=group_stat_filtered, y='step_inc', x='predatorType', hue='risky_wpair', palette=risk_palette)

# #run a anova - no obvious difference between predator type and riskiness
# group_stat_filtered['abs_step_inc'] = np.abs(group_stat_filtered['step_inc'])
# model = smf.ols('abs_step_inc ~ C(predatorType) + C(risky_wpair)', data=group_stat_filtered)
# anova_table = sm.stats.anova_lm(model.fit(), typ=2)
# print(anova_table)

# #then t-test
# print(ss.ttest_ind(group_stat_filtered.query('risky_wpair=="risk-averse"')['abs_step_inc'], 
#                    group_stat_filtered.query('risky_wpair=="risk-prone"')['abs_step_inc'],))
# # print(ss.ttest_1samp(group_stat_filtered.query('risky_wpair=="risk-averse"')['step_inc'], 0))
# # sample =group_stat_filtered.query('risky_wpair=="risk-prone"')['step_inc']
# # mean_diff = 0 - np.mean(sample)  # or just np.mean(sample)
# # std_dev = np.std(sample, ddof=1)  # use ddof=1 for sample std
# # cohens_d = mean_diff / std_dev
# # print(f"Cohen's d for risk-averse: {cohens_d}")

# # print(ss.ttest_1samp(group_stat_filtered.query('risky_wpair=="risk-prone"')['step_inc'], 0))
# # sample = group_stat_filtered.query('risky_wpair=="risk-prone"')['step_inc']
# # mean_diff = 0 - np.mean(sample)  # or just np.mean(sample)
# # std_dev = np.std(sample, ddof=1)  # use ddof=1 for sample std
# # cohens_d = mean_diff / std_dev
# # print(f"Cohen's d for risk-prone: {cohens_d}")

In [ ]:
plt.figure(figsize=(12, 4))
sns.lmplot(data = group_stat, x='individual', y='step_inc', hue='predator',
           scatter_kws={'s':10, 'alpha':0.5})
plt.ylabel('Step (group) - Step (individual)')
plt.xlabel('individual step')
print(ss.pearsonr(group_stat['individual'], group_stat['step_inc']))
#sns.regplot(data = g.loc[g.predatorType==1], x='individual', y='grp_idv_step_diff')

In [ ]:
# sns.lmplot(data = group_stat, y='jointMoney', x='reward', hue='predatorType')
# plt.plot([-20, 40], [-20, 80], ls='--', color='gray')

# # in most groups, reward is more than jointMoney/2

In [ ]:
group_stat['self_partner_risk_diff'] = group_stat.groupby(['room', 'predatorType'])['individual'].diff()
group_stat['self_partner_risk_diff'] = group_stat.groupby(['room', 'predatorType'])['self_partner_risk_diff'].transform(lambda x: x.fillna(-x.iloc[-1]))

# sns.regplot(data = g.query('predatorType==0'), x='self_partner_risk_diff', y='step_inc',
#            scatter=False, label='safe')
# sns.regplot(data = g.query('predatorType==1'), x='self_partner_risk_diff', y='step_inc',
#            scatter=False, label='dangerous')
# plt.ylabel('step (dyad) - step (individual)')
# plt.xlabel('player-partner individual step difference')
myfig = sns.lmplot(data = group_stat, x='self_partner_risk_diff', y='step_inc', hue='predator', 
           scatter_kws={'s':10, 'alpha':0.5}, legend=False, 
           aspect=1.25, height=6) #aspect = width / height
plt.legend(title='predator', fontsize=18, title_fontsize=19, markerscale=2, loc='upper right')

# set x, y label
# plt.ylabel('$player_{dyad} - player_{individual}$')
# plt.xlabel('$player_{individual} - partner_{individual}$')
plt.ylabel('Average player increase in riskiness', fontsize=22)
plt.xlabel('Average player-partner risk difference', labelpad=10, fontsize=22)

# Add a full border (spines) around the plot
for ax in myfig.axes.flat:
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

#test significance
r0, p0 = ss.pearsonr(group_stat.query('predatorType==0')['self_partner_risk_diff'], 
                     group_stat.query('predatorType==0')['step_inc'])
r1, p1 = ss.pearsonr(group_stat.query('predatorType==1')['self_partner_risk_diff'], 
                     group_stat.query('predatorType==1')['step_inc'])

# Annotate plot
plt.ylim([-8, 8])
plt.annotate(f"$r={round(r0, 2)}, p<0.001$", (-6, -6.3), fontsize=16, color='#1F77B4')
plt.annotate(f"$r={round(r1, 2)}, p<0.001$", (-6, -7.2), fontsize=16, color='#FF7F0E')

# add arrows below
ft=12.5
myoffset = 0.04
#for x-axis
y_level =-0.18
plt.annotate('', xy=(0, y_level), xytext=(0.5, y_level), #from (x,y) to (xtext, ytext) relateive to (1,1) picture
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops
            =dict(arrowstyle='<->', lw=1.5))
plt.annotate('', xy=(0.5, y_level), xytext=(1, y_level), 
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='<->', lw=1.5))
plt.annotate('player less risky than partner in idv', xy=(0.03, y_level-myoffset), 
             xycoords='axes fraction', ha='left', fontsize=ft)
plt.annotate('player riskier than partner in idv', xy=(0.56, y_level-myoffset), 
             xycoords='axes fraction', ha='left', fontsize=ft)
#for y-axis
x_lvl = -0.16
plt.annotate('', xy=(x_lvl, 0.5), xytext=(x_lvl, 1), 
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='<->', lw=1.5))
plt.annotate('', xy=(x_lvl, 0.5), xytext=(x_lvl, 0), 
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='<->', lw=1.5))
plt.annotate('player takes more risks in dyad', xy=(x_lvl-myoffset, 0.54), 
             xycoords='axes fraction', ha='left', fontsize=ft, rotation=90)
plt.annotate('player takes fewer risks in dyad', xy=(x_lvl-myoffset, -0.02),
             xycoords='axes fraction', ha='left', fontsize=ft, rotation=90)
if save: 
    plt.savefig(f'../figs/{folder}/grp_reg_{folder}.png', 
                bbox_inches='tight', dpi=200)

In [ ]:
#compromise is independent of performance ..?
sns.lmplot(data=group_stat, y='step_inc', x='reward', hue='risky_wpair', 
           scatter_kws={'s':10, 'alpha':0.5}, legend=None)
plt.legend(title='within dyad', bbox_to_anchor=(0.6, 0.75), fontsize=14, title_fontsize=15)
#test significance
print(ss.pearsonr(group_stat.query('risky_wpair=="risk-averse"')['reward'], 
                    group_stat.query('risky_wpair=="risk-averse"')['step_inc']))
print(ss.pearsonr(group_stat.query('risky_wpair=="risk-prone"')['reward'], 
                    group_stat.query('risky_wpair=="risk-prone"')['step_inc']))

In [ ]:
#reward increase by riskiness and predator
plt.figure(figsize=(3.75, 6.5))
group_stat['step_inc_towards_partner']=group_stat.apply(lambda row: row['step_inc'] if row['risky_wpair']=="risk-averse" else -row['step_inc'], axis=1)
group_stat['compromised'] = group_stat['step_inc_towards_partner']>0
group_stat['predator'] = group_stat['predatorType'].apply(lambda x: 'low-threat' if x==0 else 'high-threat')
# sns.swarmplot(data = group_stat, x='compromised', y='reward_inc', hue='risky_wpair', dodge=True, alpha=0.4, s=2,
#               legend=False)
sns.boxplot(data = group_stat, x='compromised', y='reward_inc',
            hue='risky_wpair', palette=risk_palette, 
            flierprops=dict(markersize=6), boxprops=dict(alpha=0.5), )
sns.stripplot(data = group_stat, x='compromised', y='reward_inc', dodge=True,
            hue='risky_wpair', palette=risk_palette, alpha=0.5
)
print("increase in reward in dyad: ",ss.ttest_1samp(group_stat['reward_inc'], 0))
# plt.legend(loc='lower center', fontsize=14, title_fontsize=15, title='riskiness within dyad')
plt.legend().remove()
plt.ylabel('Change in reward')
plt.xlabel('Compromised')
# plt.ylabel(r'$reward_{dyad}$ - $reward_{idv}$')
plt.axhline(y=0, ls='--', color='black', lw=1)

if save:
    plt.savefig(f'../figs/{folder}/reward_by_compromise{folder}.png', 
            bbox_inches='tight', dpi=200)


In [ ]:

#run regression
# model = smf.ols('reward_inc ~ C(risky_wpair) * C(compromised)', data=group_stat).fit()
reward_model = smf.ols('reward_inc ~ self_partner_risk_diff * step_inc_towards_partner', 
                           data=group_stat).fit()
# model = smf.ols('reward_inc ~ C(predator) * C(compromised)', data=group_stat).fit()

with open(f'../figs/{folder}/reward_reg_summary_continuous_{folder}.txt', "w") as f:
    f.write(reward_model .summary().as_text())

reward_formula = 'reward_increase ~ 1 + self_partner_risk_diff * step_inc_towards_partner'
title = "Regression Result for reward increase"
save_reg_to_table(reward_model , 4, reward_formula, title, folder)
# ss.ttest_ind(group_stat.query('risky_wpair=="risk-prone" and predatorType==0')['reward_inc'], 
#                group_stat.query('risky_wpair=="risk-prone" and predatorType==1')['reward_inc'])

In [ ]:
# regress delta_reward on cumulative_reward
# d = group_stat.groupby(['room', 'predatorType'], as_index=False)[['reward', 'reward_inc']].mean()


# sns.lmplot(data=group_stat, y='reward_inc', x='reward', hue='risky_wpair', 
#            scatter_kws={'s':10, 'alpha':0.5}, legend=None)
# sns.lmplot(data=d, y='reward_inc', x='reward', hue='predatorType', 
        #    scatter_kws={'s':10, 'alpha':0.5}, legend=None)
plt.hist(group_stat.query('predatorType==0')['self_partner_risk_diff'], alpha=0.5)
plt.hist(group_stat.query('predatorType==1')['self_partner_risk_diff'], alpha=0.5)

# maybe: dyads with higher risk diff compromise

In [ ]:
# take a look at rt
# d = df_group.query('step_rt<8').groupby(['subID', 'playerStep'], as_index=False)['step_rt'].mean()
# sns.lineplot(data = d, x='playerStep', y='step_rt')

df_group['abs_player_pred_diff']= np.abs(df_group['prediction'] - df_group['playerStep'])
df_group['log_step_rt'] = np.log(df_group['step_rt'])
d = df_group.query('step_rt<8 and abs_player_pred_diff<10').groupby(['subID', 'abs_player_pred_diff'], as_index=False)['log_step_rt'].mean()
sns.lineplot(data = d, x='abs_player_pred_diff', y='log_step_rt')
plt.xlabel('| player step - predicted partner step |')
plt.ylabel('log (RT)')

In [ ]:
# run regression on RT
# higher self-prediction discrepency -> longer RT
rt_reg = df_group.query('step_rt<8 and prediction>0').copy()
rt_reg['abs_playerStep_prediction_diff'] = np.abs(rt_reg['playerStep'] - rt_reg['prediction'])
rt_reg['log_step_rt'] = np.log(rt_reg['step_rt'])
rt_model = smf.mixedlm('log_step_rt ~ abs_playerStep_prediction_diff + playerStep', 
                    data = rt_reg,
                    groups = rt_reg['subID'],
                    re_formula = '~ abs_playerStep_prediction_diff + playerStep'
                ).fit()
print(rt_model.summary())

with open(f'../figs/{folder}/rt_reg_summary_{folder}.txt', "w") as f:
    f.write(rt_model.summary().as_text())


In [ ]:
rt_formula = '  log_step_rt ~ 1 + abs_playerStep_prediction_diff + playerStep + (1 + abs_playerStep_prediction_diff + playerStep | subID)'
title = "Regression Result for Step RT"
save_reg_to_table(rt_model, 3, rt_formula, title, folder)

In [ ]:
# correlations - no diff early and later
# corr_df1 = (
# df_group.query('trial <= 10 and playerID == 1')
#     .groupby('room')
#     .apply(lambda g: g['playerStep'].corr(g['partnerStep']))
#     .reset_index(name='corr_player_partner')
# )

# corr_df2 = (
# df_group.query('trial > 20 and playerID == 1')
#     .groupby('room')
#     .apply(lambda g: g['playerStep'].corr(g['partnerStep']))
#     .reset_index(name='corr_player_partner')
# )
# corr_df = pd.merge(corr_df1, corr_df2, on=['room'], suffixes=("_early", "_later")).dropna()
# print(ss.ttest_rel(corr_df['corr_player_partner_early'], corr_df['corr_player_partner_later']))

In [ ]:
# check prediction error between prediction and partner step. compare people with high vs low prediction error
# df_group['pred_error_bin'] = pd.qcut(df_group['pred_error'], q=3, labels=['low', 'med', 'high'])
# d = df_group.groupby('subID')['pred_error'].mean().reset_index()
plt.figure(figsize=(5.5, 4))
plt.hist(df_group['pred_error'], color='black', alpha=0.7, density=True)
plt.xlabel('absolute prediction error')
plt.ylabel('frequency of trials')

d = df_group.groupby('subID')['pred_error'].mean().reset_index() 
print(np.median(d['pred_error']), np.percentile(d['pred_error'], [25, 75]))


# if save:
#     plt.tight_layout()
#     plt.savefig(f'../figs/{folder}/pred_error_{folder}.png', 
#             bbox_inches='tight', dpi=200)

In [ ]:
# d = (df_group.query('step_rt<8 and selfBlame>-1').groupby('subID')\
#     [['pred_error', 'player_partner_diff', 'selfBlame']].mean().reset_index())
# d['player_partner_diff'] = np.abs(d['player_partner_diff'])
# sns.regplot(data = d, x='pred_error', y='selfBlame')
# ss.pearsonr(d['pred_error'], d['selfBlame'])

In [ ]:
d = df_group.query('selfBlame>=0').groupby('subID')[['pred_error', 'player_partner_diff']].mean().reset_index()
d = pd.merge(d, get_ego_bias(df_group)[['subID', 'ego_bias']], on='subID')
m_blame = smf.mixedlm(
    # "player_future_change ~ attack * selfBlame + player_partner_diff * selfBlame + attack * player_partner_diff",
    # "player_future_change ~ selfBlame * attack * player_partner_diff",
    "ego_bias ~ player_partner_diff + pred_error",  # main effects + all interactions
    data=d, 
    groups = d['subID'],
    # re_formula="~attack + player_partner_diff + selfBlame",  # only random intercept here
    # re_formula="~attack * selfBlame + player_partner_diff * selfBlame + attack * player_partner_diff",  # random intercept + random slopes for attack and player_partner_diff
    re_formula="~player_partner_diff + pred_error",  # random intercept + random slopes for player_partner_diff and pred_error
).fit(reml=False, maxiter=5000, method='lbfgs')
print(m_blame.summary())

In [ ]:
d2 = (
    (df_group['prediction'] - df_group['partnerStep'])
    .abs()
    .groupby(df_group['subID'])
    .mean()
    .reset_index(name='pred_error')
)
d = pd.merge(df_group.groupby(['subID'])['player_partner_diff'].mean().reset_index(), 
                            get_ego_bias(df_group, groupby_cols=['subID'])[['subID', 'ego_bias']])
d = pd.merge(d, d2)
# sns.regplot(data = d, x ='ego_bias', y= 'pred_error')
# print(ss.pearsonr(d['ego_bias'], d['pred_error']))

# if save the plot, input folder
partial_corr_manual(d, 'ego_bias', 'pred_error', 'player_partner_diff', plot=True, folder=folder, 
                    yname='partner step prediction error\n', xname='egocentric bias')

                

In [ ]:
# no correlation between blame (agency) and prediction error
d = df_group.query('selfBlame>-1').groupby(['subID'])[['player_partner_diff', 'selfBlame']].mean().reset_index()
d2 = (
    (df_group['prediction'] - df_group['partnerStep'])
    .abs()
    .groupby(df_group['subID'])
    .mean()
    .reset_index(name='pred_error')
)
d = pd.merge(d, d2)
# sns.regplot(data = d, x ='ego_bias', y= 'pred_error')
# print(ss.pearsonr(d['ego_bias'], d['pred_error']))

# if save the plot, input folder
partial_corr_manual(d, 'selfBlame', 'pred_error', 'player_partner_diff', plot=True, folder='', 
                    yname='partner step prediction error\n', xname='blame / credit attributed to self\n')


## trial by trial compromise

In [ ]:
df_group['outcome'] = df_group.apply(lambda row: 'harvest' if (row['attack']==0) else 'captured', axis=1).astype('category')
df_group['outcome'] = df_group['outcome'].cat.reorder_categories(['harvest', 'captured'], ordered=True)

df_cleaned = df_group.sort_values(by=['room', 'block', 'trial'])
df_cleaned['predator'] = df_cleaned['predatorType'].apply(lambda x: 'safe' if x==0 else 'dangerous')
df_cleaned['partner_changed'] = df_cleaned['partnerStep'] - df_cleaned.groupby(['subID', 'block'])['partnerStep'].shift()
df_cleaned['partner_changed_rel'] = df_cleaned['partner_changed'] * np.sign(df_cleaned.groupby(['subID', 'block'])['player_partner_diff'].shift())
df_cleaned['player_change'] = df_cleaned.groupby(['subID', 'block'])['playerStep'].shift(-1) -df_cleaned['playerStep']
df_cleaned['future_step_rt'] = df_cleaned.groupby(['subID', 'block'])['step_rt'].shift(-1)
df_cleaned['player_change_rel'] = -df_cleaned['player_change'] * np.sign(df_cleaned['player_partner_diff'])
df_cleaned = df_cleaned.query('player_change<=8 and player_change>=-8 and player_partner_diff<=8 and player_partner_diff>=-8')
# sns.lmplot(data=df_cleaned, x='partner_changed_rel', y='player_future_change_rel', scatter=False)
# sns.lineplot(data=df_cleaned.groupby(['sub', 'player_partner_diff', 'predator'], as_index=False)['player_change'].mean(), 
#              x='player_partner_diff', y='player_change', hue='predator', hue_order=['safe', 'dangerous'])
# plt.ylim([-6, 6])
# plt.xlabel('player partner step difference')
# plt.ylabel('player step change')


##small plot
plt.figure(figsize=(6.5, 5.5))

sns.lineplot(data=df_cleaned.groupby(['subID', 'player_partner_diff', 'attack'],
                                     as_index=False)['player_change'].mean(), 
             x='player_partner_diff', y='player_change', hue='attack', 
             linewidth=3,hue_order=[0, 1], palette=outcome_colors,
            )
              
##scatter on top
sns.scatterplot(data=df_cleaned.groupby(['subID', 'player_partner_diff', 'attack'],
                                     as_index=False)['player_change'].mean(),
                x='player_partner_diff',
                y='player_change', hue='attack',
                hue_order=[0, 1], palette=outcome_colors,
                s=10, alpha=0.2)

               
plt.ylim([-8.5, 8.5])
plt.xlim([-8.5, 8.5])
# plt.xlabel('player partner step difference')
# plt.ylabel('player step change')
plt.axhline(0, color='grey', linewidth=1, linestyle='--')
plt.axvline(0, color='grey', linewidth=1, linestyle='--')


##change legend labels by creating two lines of blue and orange
legend_handles=[Line2D([0], [0], color= outcome_colors[0], lw=3),
                Line2D([0], [0], color=outcome_colors[1], lw=3)]

plt.legend(legend_handles, ['harvest', 'captured'], title='outcome', 
           bbox_to_anchor=(0.65, 0.7), fontsize=16, 
           frameon=False,
           title_fontsize=17)



##change x, y labels
# plt.legend().remove()
plt.xlabel(r'$S_{player, t-1} - S_{partner, t-1}$', fontsize=22, labelpad=5)
plt.ylabel(r'$S_{player, t} - S_{player, t-1}$', fontsize=22)

if save:
    plt.savefig(f'../figs/{folder}/trial_reg2_{folder}.png', 
            bbox_inches='tight', dpi=200)

In [ ]:
df_cleaned.to_csv(f'reg_{folder}.csv')

In [ ]:
# run regression
df_change_reg = df_cleaned.query('trial>1 and trial<30 and step_rt<8 and future_step_rt<8') #remove first and last trial and auto choices
df_change_reg = df_change_reg[['player_change', 'player_partner_diff', 'outcome', 'partner_changed', 'subID', 'trial', 'block', 'partnerStep', 'playerStep']].dropna() #remove first and last trial
print(len(df_change_reg['subID'].unique()))

change_model = smf.mixedlm('player_change ~ player_partner_diff * outcome + partner_changed', 
                    data = df_change_reg,
                    groups = df_change_reg['subID'],
                    re_formula = '~ player_partner_diff * outcome + partner_changed'
                ).fit()
print(change_model.summary())

with open(f'../figs/{folder}/stepchange_reg_summary_{folder}.txt', "w") as f:
    f.write(change_model.summary().as_text())

In [ ]:
change_formula = 'player_change ~ 1 + player_partner_diff * outcome + partner_changed + (1 + player_partner_diff * outcome + partner_changed | subID)'
title = "Regression Result for Player Step Change"
save_reg_to_table(change_model, 5, change_formula, title)

In [ ]:
# additional regression controlling for player step
df_change_reg = df_cleaned.query('step_rt<8 and future_step_rt<8') #remove first and last trial and auto choices
# df_change_reg = pd.merge(df_change_reg, df_change_reg.groupby(['subID', 'predatorType'], as_index=False)['playerStep'].mean().rename({'playerStep':'avg_playerStep'}, axis=1))
# df_change_reg['mc_playerStep'] = df_change_reg['playerStep'] - df_change_reg['avg_playerStep']
df_change_reg = df_change_reg[['player_change', 'player_partner_diff', 'outcome', 'partner_changed', 'subID', 'trial', 'predatorType', 'playerStep', 'partnerStep']].dropna() #remove first and last trial
df_change_reg['playerStep_centered'] = df_change_reg['playerStep'] - 4.5 #center this
change_model2 = smf.mixedlm('player_change ~ playerStep_centered + player_partner_diff * outcome + partner_changed', 
                    data = df_change_reg,
                    groups = df_change_reg['subID'],
                    re_formula = '~ playerStep_centered + player_partner_diff * outcome + partner_changed'
                ).fit(reml=False, method="lbfgs", maxiter=2000)
print(change_model2.summary())

change_formula = 'player_change ~ 1 + playerStep_centered + player_partner_diff * outcome + partner_changed \n+ (1 + playerStep_centered + player_partner_diff * outcome + partner_changed | subID)'
title = "Additional Regression Result for Player Step Change\n"
save_reg_to_table(change_model2, 6, change_formula, title)

In [ ]:
# test for reciprocity: if partner moves towards player on the previous round, will player move towards partner on the next round?

test = df_cleaned.copy()
# binarize
test['partner_changed'] = test['partner_changed_rel'] >0 
test['player_change'] = test['player_change_rel'] >0 

d = test.groupby(['subID', 'partner_changed'], as_index=False)['player_change'].mean()
sns.pointplot(data = d, x='partner_changed', y='player_change')
plt.ylabel('player moves toward partner')
plt.xlabel('partner moved towards player')
ss.ttest_ind(d.query('partner_changed==True')['player_change'], d.query('partner_changed==False')['player_change'])

In [ ]:
ego = get_ego_bias(df_group, groupby_cols=['subID'])
median_bias = np.median(ego['ego_bias'])
print(f"median bias = {median_bias}")
ego.query('ego_bias>@median_bias')

In [ ]:
df_group = df_group.sort_values(by=['room', 'predatorType', 'trial'])
df_group['partner_changed'] = df_group['partnerStep'] - df_group.groupby(['subID',  'predatorType'])['partnerStep'].shift().fillna(0)

# df_group['partner_changed_rel'] = df_group['partner_changed'] * np.sign(df_group.groupby(['sub',  'predatorType'])['player_partner_diff'].shift().fillna(0))

# df_group['prev_attack'] = df_group.groupby(['sub',  'predatorType'])['attack'].shift()
##partner changed rel in last move
# df_group['partner_changed_rel_last'] = df_group.groupby(['sub',  'predatorType'])['partner_changed_rel'].shift(-1).fillna(0)

##repet for playerStep and partnerStep
df_group['player_future_change'] = df_group.groupby(['subID',  'predatorType'])['playerStep'].shift(-1) - df_group['playerStep']
df_group['future_step_rt'] = df_group.groupby(['subID',  'predatorType'])['step_rt'].shift(-1) 
df_group['player_changed'] = df_group['playerStep'] - df_group.groupby(['subID',  'predatorType'])['playerStep'].shift().fillna(0)

# df_group['player_changed_rel'] = - df_group['player_changed'] * np.sign(df_group.groupby(['sub',  'predatorType'])['player_partner_diff'].shift().fillna(0))
d = df_group.query('selfBlame>-1 and trial<30 and future_step_rt<8')
d['biased'] = d['subID'].apply(lambda x: True if x in ego.query('ego_bias>@median_bias')['subID'] else False)


## what if we run on subjects with higher ego bias
d = d.dropna(subset = ['player_future_change', 'outcome', 'player_partner_diff'])
m2 = smf.mixedlm(
    "player_future_change ~ outcome * selfBlame * player_partner_diff",
    # "player_future_change ~ selfBlame * attack * player_partner_diff",
    data=d, 
    groups = d['subID'],
    # re_formula="~attack + player_partner_diff + selfBlame",  # only random intercept here
    re_formula="~outcome * selfBlame * player_partner_diff",  # random intercept + random slopes for attack and player_partner_diff
).fit()
print(m2.summary())

# fig 4 Group dynamics clustering

In [ ]:
##expand g so that grp_idv_step_diff is split into two columns based on order

#filter groups by riskiness
group_stat_filtered = filter_groups_by_riskiness(group_stat)
# group_stat_filtered['grp_idv_step_diff_risk_prone'] = None
# group_stat_filtered['grp_idv_step_diff_risk_averse'] = None

##take absolute of self_partner_risk_diff 
group_stat_filtered['idvDiff'] = np.abs(group_stat_filtered['self_partner_risk_diff'])

# Populate the columns based on the 'order' column
group_stat_filtered.loc[group_stat_filtered['risky_wpair'] == 'risk-prone', 'grp_idv_step_diff_risk_prone'] = group_stat_filtered['step_inc']
group_stat_filtered.loc[group_stat_filtered['risky_wpair'] == 'risk-averse', 'grp_idv_step_diff_risk_averse'] = group_stat_filtered['step_inc']

# Group by `sub` and `predatorType`, and aggregate
collapsed_g = group_stat_filtered.groupby(['room', 'predatorType']).agg({
    'individual': 'mean',  # or use 'mean'/'sum' if needed
    'idvDiff': 'first',  # or use 'mean'/'sum' if needed
    'reward': 'sum',  # or use 'mean'/'sum' if needed
    'jointMoney': 'mean',  # or use 'mean'/'sum' if needed
    'finalStep': 'first',  # or use 'mean'/'sum' if needed
    'group': 'first',  # or use 'mean'/'sum' if needed
    'grp_idv_step_diff_risk_prone': 'sum',  # Summing non-null values per group
    'grp_idv_step_diff_risk_averse': 'sum',  # Summing non-null values per group
    'predator': 'first'  # or use 'mean'/'sum' if needed
}).reset_index()

# Drop rows with NaN in both `grp_idv_step_diff_risk_prone` and `grp_idv_step_diff_risk_averse`
# collapsed_g.dropna(subset=['grp_idv_step_diff_risk_prone', 'grp_idv_step_diff_risk_averse'], how='all', inplace=True)

# Ensure that `grp_idv_step_diff_risk_prone` and `grp_idv_step_diff_risk_averse` columns are numeric
collapsed_g['grp_idv_step_diff_risk_prone'] = pd.to_numeric(collapsed_g['grp_idv_step_diff_risk_prone'], errors='coerce')
collapsed_g['grp_idv_step_diff_risk_averse'] = pd.to_numeric(collapsed_g['grp_idv_step_diff_risk_averse'], errors='coerce')

# Fill NaN values if needed (e.g., with 0 or another value)
# collapsed_g['grp_idv_step_diff_risk_prone'].fillna(0, inplace=True)
# collapsed_g['grp_idv_step_diff_risk_averse'].fillna(0, inplace=True)

## add a new column based on diff - low or high - based on median 

# Calculate the median of the 'diff' column
median_diff = collapsed_g['idvDiff'].median()

collapsed_g['diffCategory'] = np.where(collapsed_g['idvDiff'] > median_diff, 'high', 'low')

# Now try plotting
sns.lmplot(data=collapsed_g, x='grp_idv_step_diff_risk_prone',
           hue='predatorType', 
           y='grp_idv_step_diff_risk_averse',  col='diffCategory',
           scatter_kws={'alpha': 0.5})

In [ ]:
#people with small risk difference compromise and big risk difference compensate?
#do clustering on this and see if it correlates with behavioral data?
num_clusters = 4 #commented method shows that 4 is best
kmeans = KMeans(n_clusters=num_clusters, n_init=1)  # Adjust the number of clusters as needed
collapsed_g['cluster'] = kmeans.fit_predict(collapsed_g[['grp_idv_step_diff_risk_prone', 'grp_idv_step_diff_risk_averse']])

# Scatter plot to show clusters
plt.figure(figsize=(8, 8))
sns.scatterplot(
    x='grp_idv_step_diff_risk_prone', 
    y='grp_idv_step_diff_risk_averse', 
    hue='cluster', 
    palette='Set2', 
    data=collapsed_g, 
    s=100, 
    alpha=0.7
)

plt.axhline(0, color='gray', ls='--')
plt.axvline(0, color='gray', ls='--')
plt.title('Scatter Plot of Clusters based on Partner Steps')
plt.legend(title='Cluster Category')
plt.grid(True)


In [ ]:
#manully separate them into 4 quadrants
def get_cluster(row):
    if row['grp_idv_step_diff_risk_prone'] >0:
        if row['grp_idv_step_diff_risk_averse'] >0:
            cluster = "risk increase"
        else:
            cluster = "risk diverge"
    else:
        if row['grp_idv_step_diff_risk_averse'] <0:
            cluster = "risk decrease"
        else:
            cluster = "risk converge"
    return cluster

collapsed_g['cluster'] = collapsed_g.apply(get_cluster, axis=1)
clusters = ['risk decrease', 'risk increase', 'risk converge', 'risk diverge']
# Scatter plot to show clusters
plt.figure(figsize=(8, 8))
sns.scatterplot(
    x='grp_idv_step_diff_risk_prone', 
    y='grp_idv_step_diff_risk_averse', 
    hue='cluster', 
    palette='Set2', 
    data=collapsed_g, 
    hue_order=clusters,
    s=60, 
    alpha=0.7
)

plt.axhline(0, color='gray', ls='--')
plt.axvline(0, color='gray', ls='--')
plt.xlabel('$\Delta$ risk (risk-prone)')
plt.ylabel('$\Delta$ risk (risk-averse)')
plt.legend(fontsize=16)
plt.grid(True, linewidth=0.5)
if save:
    plt.savefig(f'../figs/{folder}/category_{folder}.png', 
            bbox_inches='tight', dpi=200)

In [ ]:
def mybar(**kwargs):
    sns.barplot(alpha=0.4, palette='Set2', order=clusters, **kwargs)

def myswarm(**kwargs):
    sns.swarmplot(palette='Set2', order=clusters, alpha=0.8, s=3, **kwargs)

In [ ]:
# Assume you already have 'collapsed_g' and 'clusters' defined
fig, ax = plt.subplots(1, 1, figsize=(5, 5.5))

# Plot swarm and barplot
myswarm(data=collapsed_g, x='cluster', y='idvDiff')
mybar(data=collapsed_g, x='cluster', y='idvDiff')

plt.xticks(rotation=30)
plt.ylabel('baseline risk difference between dyad')
plt.xlabel('')
plt.ylim([0, 7.5])

# Run ANOVA
model = smf.ols('idvDiff ~ C(cluster)', data=collapsed_g).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

# # --- Create combined groups ---
# # Define which clusters go into each combined group
group1_clusters = ['risk decrease', 'risk increase']  # <-- replace with your actual cluster labels
group2_clusters = ['risk converge', 'risk diverge']  # <-- replace with your actual cluster labels

# Create a temporary DataFrame with relabeled groups
collapsed_g_combined = collapsed_g.copy()
collapsed_g_combined['comparison_group'] = collapsed_g_combined['cluster'].apply(
    lambda x: 'Group1+2' if x in group1_clusters else ('Group3+4' if x in group2_clusters else 'Other')
)
t, p = ss.ttest_ind(collapsed_g_combined.query('comparison_group=="Group1+2"')['idvDiff'],
            collapsed_g_combined.query('comparison_group=="Group3+4"')['idvDiff'])
print(f"t = {t}, p = {p}")

y_h = 6.9
y_l = 6.6
# Annotate the plot
ax.text(x=1.5, y=y_h, s=get_sig(p),
        ha='center', va='bottom',
        fontsize=16, color='black')
ax.plot([0.5, 2.5], [y_h, y_h], color='black', linewidth=1.5)
ax.plot([0, 1], [y_l, y_l], color='black', linewidth=1.5)
ax.plot([2, 3], [y_l, y_l], color='black', linewidth=1.5)
ax.plot([0.5, 0.5], [y_l, y_h], color='black', linewidth=1.5)
ax.plot([2.5, 2.5], [y_l, y_h], color='black', linewidth=1.5)
if save:
    plt.savefig(f'../figs/{folder}/risk_diff_by_category_{folder}.png', 
            bbox_inches='tight', dpi=200)

In [ ]:
sns.set_context("talk")
# fig, ax = plt.subplots(1, 1, figsize=(5, 5.5))
# # sns.boxplot(data = collapsed_g, x='cluster', y='idvDiff', palette='Set2')
# sns.swarmplot(data = collapsed_g, x='cluster', y='idvDiff', 
#               palette='Set2', alpha=0.8, order=clusters)
# sns.barplot(data = collapsed_g, x='cluster', y='idvDiff', 
#             palette='Set2', alpha=0.4,  order=clusters)
# plt.xticks(rotation=30)
# plt.ylabel('baseline risk difference between dyad')
# plt.xlabel('')
# model = smf.ols('idvDiff ~ C(cluster)', data=collapsed_g).fit()
# anova_table = sm.stats.anova_lm(model, typ=2)
# print(anova_table)

# #annotate?
# pairs = [(clusters[i], clusters[j]) for i in range(len(clusters)) for j in range(i)]
# annotator = Annotator(ax, pairs, data=collapsed_g, x='cluster', y='idvDiff', 
#                       order=clusters)
# #set configurations and apply
# annotator.configure(test='t-test_ind', hide_non_significant=True,
#                     # comparisons_correction="bonferroni", 
#                     text_format='star', loc='inside', 
#                     line_height =0, text_offset=-1)#for style
# annotator.apply_and_annotate()


# sns.swarmplot(data = collapsed_g, x='cluster', y='idvDiff', palette='Set2', alpha=0.7)
fig, ax = plt.subplots(1, 1, figsize=(5, 5.5))
myswarm(data = collapsed_g, x='cluster', y='individual')
mybar(data = collapsed_g, x='cluster', y='individual')
plt.xticks(rotation=30)
plt.ylabel('average baseline riskiness of dyad')
plt.xlabel('')
model = smf.ols('individual ~ C(cluster)', data=collapsed_g).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)


#annotate?
pairs = [("risk increase", "risk decrease"), ("risk converge", "risk diverge"),]
annotator = Annotator(ax, pairs, data=collapsed_g, x='cluster', y='individual', 
                      order=clusters)
#set configurations and apply
annotator.configure(test='t-test_ind', hide_non_significant=True,
                    # comparisons_correction="bonferroni", 
                    text_format='star', loc='inside', 
                    line_height =0, text_offset=-1)#for style
annotator.apply_and_annotate()

# if save:
#     plt.savefig(f'../figs/{folder}/avg_risk_by_category_{folder}.png', 
#             bbox_inches='tight', dpi=200)

In [ ]:
#further: who drives this increase or decrease?

plt.figure(figsize=(5, 6))
test = pd.merge(group_stat, collapsed_g[['room', 'cluster']].drop_duplicates())
print(len(test))
sns.boxplot(data = test, x='cluster', y='step_inc', palette=risk_colors, 
              hue='risky_wpair', order=clusters,
              width=0.7, linewidth=0.7, flierprops=dict(marker='o', markersize=4, alpha=0.5))
# sns.barplot(data = test, x='cluster', y='step_inc', palette='Set2', 
#               hue='risky_wpair', alpha=0.5)
plt.legend(title = 'within dyad', loc='upper right', 
           fontsize=13, title_fontsize=14, handletextpad=0.5,handlelength=1)
# sns.barplot(data = collapsed_g, x='cluster', y='idvDiff', palette='Set2', alpha=0.4)
plt.xticks(rotation=30)
plt.ylabel('average increase in steps')
plt.axhline(0, color='black', ls='--', lw=0.75)
plt.xlabel('')
plt.ylim([-6.3, 6.3])
if save:
    plt.savefig(f'../figs/{folder}/step_increase_by_category_{folder}.png', 
            bbox_inches='tight', dpi=200)
# model = smf.ols('idvDiff ~ C(cluster) + C(risky_wpair)', data=collapsed_g).fit()
# anova_table = sm.stats.anova_lm(model, typ=2)
# print(anova_table)

model = smf.ols('step_inc ~ C(cluster) * C(risky_wpair)', data=test).fit()
print(model.summary())

In [ ]:
print(ss.ttest_ind(collapsed_g.query('cluster=="risk converge"')['grpIdvRewardDiff'], 
                   collapsed_g.query('cluster=="risk diverge"')['grpIdvRewardDiff']))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 6))
collapsed_g['grpIdvRewardDiff']= collapsed_g['jointMoney'] - collapsed_g['reward']
mybar(data = collapsed_g, x='cluster', y='grpIdvRewardDiff')
myswarm(data = collapsed_g, x='cluster', y='grpIdvRewardDiff')
# sns.boxplot(data = collapsed_g, x='cluster', y='grpIdvRewardDiff', palette='Set2')
plt.ylabel('average $\Delta$ reward')
plt.xticks(rotation=30)
plt.ylim([-45, 55])
plt.xlabel('')


for i, subset in collapsed_g.groupby('cluster'):
    t, p = ss.ttest_1samp(subset['grpIdvRewardDiff'], 0)
    # Print the t-statistic and p-value
    print(f"Cluster {i}: t-statistic = {t}, p-value = {p}")
    #annotate
    idx = collapsed_g['cluster'].unique().tolist().index(i)
    #correct for multiple comparison
    ax.text(x=idx, y=47, s=get_sig(p),  ha='center', va='center')
    # ax.plot([idx-0.25, idx+0.25], [45, 45], color='black', linewidth=1.5)
if save:
    plt.savefig(f'../figs/{folder}/reward_inc_by_category_{folder}.png', 
            bbox_inches='tight', dpi=200)

In [ ]:
d = pd.merge(group_stat[['subID','room', 'predatorType', 'risky_wpair', 'reward_inc']], 
             collapsed_g[['room', 'predatorType', 'cluster']])
# ss.ttest_ind(d.query('cluster=="risk converge" and risky_wpair=="risk-averse"')['reward_inc'],
#              d.query('cluster=="risk increase" and risky_wpair=="risk-averse"')['reward_inc'])

# ss.ttest_ind(d.query('cluster=="risk converge" and risky_wpair=="risk-prone"')['reward_inc'],
#              d.query('cluster=="risk increase" and risky_wpair=="risk-prone"')['reward_inc'])

In [ ]:
# can you just regress risk increase on 

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(5, 5.5))
# sns.barplot(data = collapsed_g, x='cluster', y='jointMoney', 
#             palette='Set2', alpha=0.4, order=clusters)
# sns.swarmplot(data = collapsed_g, x='cluster', y='jointMoney', 
#               palette='Set2', alpha=0.6, order=clusters)
# # sns.boxplot(data = collapsed_g, x='cluster', y='grpIdvRewardDiff', palette='Set2')
# plt.ylabel('dyad reward')
# plt.xticks(rotation=30)
# # plt.ylim([-42, 42])


# #annotate?
# annotator = Annotator(ax, pairs, data=collapsed_g, x='cluster', y='individual', 
#                       order=clusters)
# #set configurations and apply
# annotator.configure(test='t-test_ind', hide_non_significant=True,
#                     comparisons_correction="bonferroni", 
#                     text_format='star', loc='inside', 
#                     line_height =0, text_offset=-1)#for style
# annotator.apply_and_annotate()

In [ ]:
#no difference in sum blame. what about blame async?
fig, ax = plt.subplots(1, 1, figsize=(5, 5.5))
test= df_group.copy().query('selfBlame!=-1 and partnerBlame!=-1 and playerID==1') # keep 1 per group
test['blameAsync'] = abs(test['selfBlame'] + test['partnerBlame'] - 1)
test = test.groupby(['room'], as_index=False)['blameAsync'].mean()
collapsed_g2 = pd.merge(test,
                collapsed_g[['room', 'cluster']].drop_duplicates())


mybar(data = collapsed_g2, x='cluster', y='blameAsync')
myswarm(data = collapsed_g2, x='cluster', y='blameAsync')
plt.xticks(rotation=30)
plt.xlabel('')


#annotate?
pairs = [("risk increase", "risk diverge"), 
         ("risk converge", "risk diverge"),
         ("risk decrease", "risk diverge"), 
        ]
annotator = Annotator(ax, pairs, data=collapsed_g, x='cluster', y='individual', 
                      order=clusters)
#set configurations and apply
annotator.configure(test='t-test_ind', hide_non_significant=False,
                    # comparisons_correction="bonferroni", 
                    text_format='star', loc='inside', 
                    line_height =0, text_offset=-1)#for style
annotator.apply_and_annotate()


In [ ]:
# self-attribution bias difference between clusters
bias_df = df_group.query('selfBlame!=-1').groupby(['attack', 'subID', 'room', 'predatorType'], as_index=False)['selfBlame'].mean()
bias_df = pd.merge(bias_df.query('attack==False'), bias_df.query('attack==True'), on=['subID', 'room', 'predatorType'])
bias_df['bias'] = bias_df['selfBlame_x'] - bias_df['selfBlame_y']
bias_df = pd.merge(bias_df[['subID', 'bias', 'room', 'predatorType']], collapsed_g)


fig, ax = plt.subplots(1, 1, figsize=(5, 5.5))
mybar(data = bias_df, x='cluster', y='bias')
myswarm(data = bias_df, x='cluster', y='bias')
# sns.stripplot(data = bias_df, x='cluster', y='bias', palette='Set2')
plt.xticks(rotation=30)
plt.xlabel('')
plt.ylabel('egocentric bias')

#annotate?
pairs = [("risk increase", "risk diverge"), 
         ("risk converge", "risk diverge"),
         ("risk decrease", "risk diverge"), 
        #  ("risk increase", "risk converge"), 
        #  ("risk decrease", "risk converge"), 
        ]
annotator = Annotator(ax, pairs, data=bias_df, x='cluster', y='bias', 
                      order=clusters)
#set configurations and apply
annotator.configure(test='t-test_ind', hide_non_significant=False,
                    # comparisons_correction="bonferroni", 
                    text_format='star', loc='inside', 
                    line_height =0, text_offset=-1)#for style
annotator.apply_and_annotate()

if save:
    plt.savefig(f'../figs/{folder}/blame_bias_by_category_{folder}.png', 
            bbox_inches='tight', dpi=200)

In [ ]:
len(bias_df)

In [ ]:
# #idv
# df_idv['player_change'] = df_idv.query('trial>60').groupby(['sub', 'predatorType'])['choice'].shift(-1) -df_idv['choice']
# # sns.lmplot(data=df_cleaned, x='partner_changed_rel', y='player_future_change_rel', scatter=False)
# sns.lineplot(data=df_idv.groupby(['sub', 'choice', 'predatorType'], as_index=False)['player_change'].mean(), 
#              x='choice', y='player_change', hue='predatorType')
# plt.ylim([-6, 6])
# # plt.xlabel('player partner step difference')
# # plt.ylabel('player step change')

In [ ]:
#what drives this correlation?
# 

# Fig2 Sharing Responsibility and Reward

In [ ]:
#plot split by step
#df_group = pd.read_csv('../processed_data/regression_group_all_data.csv', index_col=[0])
#df_group['step_diff'] = df_group['stepChoice'] - df_group['partnerStep']
# df_group['playerKeep2'] = df_group.apply(lambda x: 1-x['playerKeep'] if x['predatorAttack'] else x['playerKeep'],
# 
# rooms = []
# d = df_group.query('room not in @rooms')#.query('subID not in @subs')

# plt.figure(figsize=(5.5, 4))
# d = d.loc[(d.playerKeep2!=-1) & (d.selfBlame !=-1)].groupby(['sub', 'attack', 'player_partner_diff'],
#                     as_index=False)[['playerKeep', 'selfBlame']].mean()
# plt.axhline(y=0.5, ls='--', color='black', alpha=0.5)
# plt.axvline(x=0, ls='--', color='black', alpha=0.5)
# sns.lineplot(data=d, 
#              x='player_partner_diff', y='playerKeep', hue='attack', palette=['green', 'red']
#         #      , err_style='bars'
#              )
# plt.xlabel('Step (player) - Step (partner)')
# plt.ylabel('player keeps')
# plt.ylim([0, 1])
# #plt.yticks([0.4, 0.6, 0.8, 1], ['40%', '60%', '80%', '100%'])
# #plt.legend().remove()
# plt.legend(loc='lower right', title='attack'
#         #    title='outcome', labels=['win', 'lose'], 
#            )#,fontsize='16')

#step prediction - #step partner

#plot split by step
#df_group = pd.read_csv('../processed_data/regression_group_all_data.csv', index_col=[0])
#df_group['step_diff'] = df_group['stepChoice'] - df_group['partnerStep']
# df_group['playerKeep2'] = df_group.apply(lambda x: 1-x['playerKeep'] if x['predatorAttack'] else x['playerKeep'],
plt.figure(figsize=(6, 5))
plt.axhline(y=0.5, ls='--', color='black', alpha=0.5)
plt.axvline(x=0, ls='--', color='black', alpha=0.5)
d = df_group.query('selfBlame!=-1').groupby(['subID','player_partner_diff', 'outcome'],
                                             as_index=False)['selfBlame'].mean()

#remove when playerStep=0

d = d.query('player_partner_diff>-9 and player_partner_diff<9')
#for plotting
# d['outcome'] = d['attack'].apply(lambda x: 'win' if x==False else 'lose')
sns.lineplot(data=d, x='player_partner_diff', y='selfBlame', hue='outcome', 
            #  palette=['green', 'red'] #'fuchsia'
            palette = outcome_colors
            )
plt.xlabel('$S_{player} - S_{partner}$',fontsize=20)
plt.ylabel('credit / blame attributed to self', fontsize=20)
plt.ylim([0, 1])
plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1], ['0%', '20%', '40%', '60%', '80%', '100%'])
#plt.legend().remove()
plt.legend(loc='lower right',
           title='outcome', frameon=False,fontsize=16, title_fontsize=17)

# add arrows below
plt.annotate('', xy=(0, -0.22), xytext=(0.5, -0.22), 
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='<->', lw=1.5))
plt.annotate('', xy=(0.5, -0.22), xytext=(1, -0.22), 
            xycoords='axes fraction', textcoords='axes fraction',
            arrowprops=dict(arrowstyle='<->', lw=1.5))
plt.annotate('self less risky than partner', xy=(0.03, -0.27), 
             xycoords='axes fraction', ha='left', fontsize=12.5)
plt.annotate('self riskier than partner', xy=(0.56, -0.27), 
             xycoords='axes fraction', ha='left', fontsize=12.5)
if save:
    plt.savefig(f'../figs/{folder}/blame_{folder}.png', 
            bbox_inches='tight', dpi=200)

In [ ]:
#for each subject, compare blame vs credit when self behind?

In [ ]:
reg_data = df_group.query('selfBlame!=-1')
reg_data = reg_data[['subID', 'player_partner_diff', 'outcome', 'selfBlame', 'trial', 'predatorType']].dropna()
# mean center blame
reg_data['selfBlame'] = reg_data['selfBlame'] - 0.5
blame_model = smf.mixedlm(formula='selfBlame ~ player_partner_diff * outcome', 
                    data=reg_data, groups = reg_data['subID'],
                    re_formula="~ player_partner_diff * outcome"
                    ).fit()

print(blame_model.summary())
with open(f'../figs/{folder}/blame_reg_summary_{folder}.txt', "w") as f:
    f.write(blame_model.summary().as_text())

In [ ]:
blame_formula = 'selfBlame ~ 1 + player_partner_diff * outcome + (1 + player_partner_diff * outcome | subID)'
title = "Regression Result for Responsibility Attribution"
save_reg_to_table(blame_model, 4, blame_formula, title)

In [ ]:
#bar plot of blame diff of each individual?
plt.figure(figsize=(4, 6))
df_group['blame_diff'] = df_group.groupby(['room', 'trial', 'predatorType'], as_index=False)['selfBlame'].diff()
df_group['blame_diff'] = np.abs(df_group['blame_diff'])
d = df_group.groupby(['outcome', 'room', 'attack'], as_index=False)['blame_diff'].mean()
#more disagreement when attack happens. why?
sns.boxplot(data = d, x='outcome', y='blame_diff', palette=outcome_palette, 
            boxprops=dict(alpha=.8))
# plt.xticks([0, 1], ['win', 'lose'])
plt.ylim(top=1)
plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1], ['0%', '20%', '40%', '60%', '80%', '100%'])
# plt.xlabel('outcome')
plt.ylabel('credit / blame difference within dyad', labelpad=1)
# run a independent t-test
d = d.dropna(subset=['blame_diff'])
r, p = ss.ttest_ind(d.query('attack==True')['blame_diff'], 
                    d.query('attack==False')['blame_diff']) 
# should you do paired t-test here? no because they are different dyads
d2 = pd.merge(d.query('attack==True'), d.query('attack==False'), on=['room'], 
              how='inner', suffixes=['_win', '_lose'])
r, p = ss.ttest_rel(d2['blame_diff_win'], d2['blame_diff_lose'])
print(f"blame difference: r={r}, p={p}")
#people are more likely to share reward than losses
# Annotate the plot with significance if p-value < 0.05
sig = get_sig(p)
plt.text(0.5, 0.89, sig, ha='center', va='bottom', fontsize=20)
plt.plot([0, 0, 1, 1], [0.9, 0.9, 0.9, 0.9], lw=1.5, color='black')
if save:
    plt.savefig(f'../figs/{folder}/blame_diff_{folder}.png', 
            bbox_inches='tight', dpi=200)


#bar plot of blame async of each dyad?
plt.figure(figsize=(4, 6))
test= df_group.sort_values(by=['room', 'predatorType', 'trial']).copy()
test['partnerBlame'] = test.groupby(['room', 'trial', 'predatorType'])['selfBlame'].shift()
test = test.dropna(subset=['partnerBlame']).query(
    'selfBlame!=-1 and partnerBlame!=-1')
test['blameAsync'] = abs(test['selfBlame'] + test['partnerBlame'] - 1)
d = test.groupby(['attack', 'room', 'outcome'], as_index=False)['blameAsync'].mean()
#more disagreement when attack happens. why?
sns.boxplot(data = d, x='outcome', y='blameAsync', palette = outcome_palette)
# plt.xticks([0, 1], ['win', 'lose'])
plt.ylim(top=1)
plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1], ['0%', '20%', '40%', '60%', '80%', '100%'])
# plt.xlabel('outcome')
plt.ylabel('credit / blame asynchrony within dyad', labelpad=1)
#run a paired t-test
d = d.dropna(subset=['blameAsync'])
# should you do paired t-test here? no because they are different dyads
d2 = pd.merge(d.query('attack==True'), d.query('attack==False'), on=['room'], 
              how='inner', suffixes=['_win', '_lose'])
r, p = ss.ttest_rel(d2['blameAsync_win'], d2['blameAsync_lose'])
print(f"blame difference: r={r}, p={p}")


if save:
    plt.savefig(f'../figs/{folder}/blame_async_{folder}.png', 
            bbox_inches='tight', dpi=200)
#people are more likely to share reward than losses
#people are more likely to share reward than losses
# Annotate the plot with significance if p-value < 0.05
sig = get_sig(p)
plt.text(0.5, 0.89, sig, ha='center', va='bottom', fontsize=20)
plt.plot([0, 0, 1, 1], [0.9, 0.9, 0.9, 0.9], lw=1.5, color='black')

In [ ]:
#count the max number of attack trials for each subject
# l= df_group.groupby(['sub', 'room'], as_index=False)['attack_count'].max()
# #select all the noattack trials to match the number of attack trials for each subject
# merged_df = pd.merge(df_group[['sub', 'room', 'noattack_count', 'selfBlame', 'blame_diff']], l)
# filtered_attack = merged_df[merged_df['noattack_count'] <= merged_df['attack_count']]
# #merge with all attack trials
# filtered_trials = pd.concat([filtered_attack.drop('attack_count', axis=1), df_group.query('attack==1')[['sub', 'room', 'selfBlame', 'attack', 'blame_diff', 'attack_count']]])
# filtered_trials['attack'] = filtered_trials['attack'].fillna(False)
# # #calculate the mean self blame for each subject for selected noattack trials
# # filtered_attack['blame_diff'] = filtered_attack.groupby('room', as_index=False)['selfBlame'].diff()
# # #calculate the mean self blame for each subject for attack trials
# # meanblame_attack['blame_diff'] = df_group.query('attack==1').groupby('room', as_index=False)['selfBlame'].diff()
# # #compare the means
# # d = pd.merge(meanblame_attack, meanblame_noattack, on='room')
# # sns.regplot(data=d, x='selfBlame_x', y='selfBlame_y')
# print(len(filtered_trials))

In [ ]:
# plt.axhline(y=0.5, ls='--', color='black', alpha=0.5)
# plt.axvline(x=0, ls='--', color='black', alpha=0.5)
# sns.lineplot(data=g.groupby(['sub', 'partner_diff', 'attack'], as_index=False)['selfBlame'].mean(),
#              x='partner_diff', y='selfBlame', label='partner', color='salmon')
# sns.lineplot(data=g.groupby(['sub', 'player_diff', 'attack'], as_index=False)['selfBlame'].mean(),
#              x='player_diff', y='selfBlame', label='player', color='teal')
# plt.xlabel('Step (t) - Step (t-1)')
# plt.ylabel('player self-blame')
# plt.legend(title='data type', loc='lower center')
# plt.ylim([0, 1])



# plt.figure()
# plt.axhline(y=0.5, ls='--', color='black', alpha=0.5)
# plt.axvline(x=0, ls='--', color='black', alpha=0.5)
# sns.lineplot(data=g2.groupby(['sub', 'partner_diff', 'attack'], as_index=False)['playerKeep'].mean(),
#              x='partner_diff', y='playerKeep', label='partner', color='salmon')
# sns.lineplot(data=g2.groupby(['sub', 'player_diff', 'attack'], as_index=False)['playerKeep'].mean(),
#              x='player_diff', y='playerKeep', label='player', color='teal')
# plt.xlabel('Step (t) - Step (t-1)')
# plt.ylabel('player keeps')
# plt.legend(title='data type', loc='lower center')
# plt.ylim([0, 1])

In [ ]:
# df_group['future_step_change'] = df_group.groupby(['subID', 'predatorType'])['playerStep'].shift(-1) - df_group['playerStep']
# test = df_group.query('selfBlame!=-1').copy()
# test['player_partner_diff'] =  np.sign(test['player_partner_diff'])  # Convert to binary
# # test['selfBlame'] = np.sign(test['selfBlame'] - 0.5)  # Convert blame to -1, 0, 1
# test['compromise'] =  np.sign(test['future_step_change']) == -np.sign(test['player_partner_diff'])  # Convert to binary
# # test['compromise'] =  - test['future_step_change'] / test['player_partner_diff']

# test=test.groupby(['predatorType', 'selfBlame', 'attack', 'sub'], as_index=False)['compromise'].mean()
# test = pd.merge(test, group_stat[['sub', 'risky_wpair', 'predatorType']])
# sns.lineplot(data = test, hue='attack', y='compromise', x='selfBlame', style = 'risky_wpair', palette=outcome_colors)
# # plt.legend(loc='upper center', title='attack')
# plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='attack')


In [ ]:
# get egocentric bias
ego_bias = get_ego_bias(df_group, groupby_cols=['subID'])
high_bias = ego_bias[ego_bias['ego_bias'] > np.median(ego_bias['ego_bias'])]


# egocentric bias by riskiness
m = pd.merge(ego_bias, df_group.groupby('subID', as_index=False)[['player_partner_diff']].mean())
r, p = ss.pearsonr(m['ego_bias'], m['player_partner_diff'])
g = sns.lmplot(data=m, y='ego_bias', x='player_partner_diff', scatter_kws={'s':30, 'alpha':0.5})
plt.ylabel('egocentric bias')
plt.xlabel('average player partner distance')
p_txt = f'= {round(p, 3)}' if p>0.001 else '< 0.001'
plt.annotate(f'r = {r:.2f}\np {p_txt}', 
             xy=(0.05, 0.82), xycoords='axes fraction', fontsize=16)
if save:
    plt.savefig(f'../figs/{folder}/ego_riskiness_{folder}.png', 
                bbox_inches='tight', dpi=200)

In [ ]:
#plot step change vs blame, separate by attack
plt.figure(figsize=(6, 5))
df_group = df_group.sort_values(by=['predatorType', 'room', 'trial', 'subID'])
df_group['future_step_change'] = df_group.groupby(['subID', 'predatorType'])['playerStep'].shift(-1) - df_group['playerStep']
df_group['future_step_rt'] = df_group.groupby(['subID', 'predatorType'])['step_rt'].shift(-1)
test = df_group.query('selfBlame!=-1').copy()
#test['player_partner_diff'] =  np.sign(test['player_partner_diff'])  # Convert to binary
# test['selfBlame'] = np.sign(test['selfBlame'] - 0.5)  # Convert blame to -1, 0, 1
# test['compromise'] = test.apply(lambda row: row['future_step_change']/row['playerStep'] if np.sign(row['future_step_change']) == -np.sign(row['player_partner_diff']) \
#                                 else -row['future_step_change']/row['playerStep'], axis=1)  # Convert to binary
test['compromise'] = np.sign(test['future_step_change']) == -np.sign(test['player_partner_diff'])  # Convert to binary
# test['compromise'] =  - test['future_step_change'] / test['player_partner_diff']
d=test.groupby(['subID', 'selfBlame', 'outcome'], as_index=False)['compromise'].mean()

sns.pointplot(data = d.query('subID in @high_bias.subID'), hue='outcome', y='compromise', x='selfBlame', 
              palette=outcome_palette, dodge=0.0, legend=None, alpha=0.7)
sns.pointplot(data = d.query('subID not in @high_bias.subID'), hue='outcome', y='compromise', x='selfBlame', 
              palette=outcome_palette, dodge=0.25, ls='--', legend=None, alpha=0.8)
legend_elements = [

    Line2D([0], [0], color='black', ls='-',  label='high'),
        Line2D([0], [0], color='black', ls='--', label='low'),
]
plt.legend(handles=legend_elements, loc='upper center', title='egocentric bias', 
        fontsize=15, title_fontsize=16)
# plt.legend().remove()
# plt.legend().remove()
plt.ylabel('P (move towads partner)', fontsize=20)
# plt.ylim([0, 1])
plt.xlabel('credit / blame attributed to self', labelpad=10, fontsize=20)
# plt.xticks([0, 0.25, 0.5, 0.75, 1], ['0%', '25%', '50%', '75%', '100%'])
plt.xticks([0, 1, 2, 3, 4], ['0%', '25%', '50%', '75%', '100%'])
if save:
    plt.savefig(f'../figs/{folder}/compromise_blame_v2_{folder}.png',
                bbox_inches='tight', dpi=200)



In [ ]:
test2 = pd.merge(test, ego_bias)
test2['high_bias'] = test2['subID'].isin(high_bias.subID).astype(int)
test2['compromise'] = test2['compromise'].astype(float)  # ensure numeric
test2 = test2.dropna(subset = ['selfBlame', 'ego_bias', 'outcome'])


# key test: does selfBlame -> compromise differ by bias group and outcome?
m = smf.mixedlm(
    "compromise ~ selfBlame * high_bias * outcome",
    data=test2.query('selfBlame != -1 and future_step_rt<8'),
    groups=test2.query('selfBlame != -1 and future_step_rt<8')['subID']
).fit()

print(m.summary())

In [ ]:
plt.hist(test.query('player_partner_diff<0 and attack==False')['selfBlame'], alpha=0.5)
plt.hist(test.query('player_partner_diff<0 and attack==True')['selfBlame'], alpha=0.5)

In [ ]:
test[['subID', 'trial', 'player_partner_diff', 'playerStep', 'future_step_change', 'attack', 'predatorType']].head(n=10)

In [ ]:
test = test.dropna(subset=['future_step_change', 'selfBlame', 'future_step_rt'])
if np.min(test['selfBlame'])>=0:
    test['selfBlame'] = test['selfBlame'] - 0.5
test['bin_player_partner_diff'] = np.sign(test['player_partner_diff'])
mra = smf.mixedlm('future_step_change ~ selfBlame * attack', 
            data=test.query('player_partner_diff<0'), 
            groups=test.query('player_partner_diff<0')['subID'],
            re_formula='selfBlame * attack'
            ).fit()
print(mra.summary())

mrp = smf.mixedlm('future_step_change ~ selfBlame * attack', 
            data=test.query('player_partner_diff>0'), 
            groups=test.query('player_partner_diff>0')['subID'],
            re_formula='selfBlame * attack'
            ).fit()
print(mrp.summary())

In [ ]:
test[['selfBlame', 'selfBlame2', 'attack']]

In [ ]:
sns.kdeplot(data =test, x='selfBlame2',hue='bin_player_partner_diff')

In [ ]:
test['selfBlame2'] = test.apply(lambda row: -row['selfBlame'] if row['attack']==True else row['selfBlame'], axis=1)
test['future_step_change_rel'] = test.apply(lambda row: row['future_step_change'] if row['player_partner_diff']<0 else -row['future_step_change'], axis=1)
# test = test.dropna(subset=['future_step_change', 'selfBlame', 'future_step_rt'])
smf.mixedlm('future_step_change ~ selfBlame2 * player_partner_diff * attack', 
            data=test.query('player_partner_diff != 0'),
            groups=test.query('player_partner_diff != 0')['subID'],
            # re_formula='selfBlame2 * player_partner_diff * attack'
            ).fit().summary()


In [ ]:
# cluster people based on risk and reward

In [ ]:
# test['future_step_change_frac'] = test['future_step_change'] / test['playerStep']
# t = test.groupby(['selfBlame', 'bin_player_partner_diff', 'subID', 'attack'], as_index=False)['future_step_change'].mean()
# sns.lineplot(data=t.query('subID in @s'), y='future_step_change', x='selfBlame', style='bin_player_partner_diff'
#             )
# plt.legend(bbox_to_anchor=(1.05, 1), title='bin_player_partner_diff')

In [ ]:
t = test.groupby(['selfBlame', 'bin_player_partner_diff', 'attack', 'subID'], as_index=False)['compromise'].mean()
sns.lineplot(data=t, x='selfBlame', y='compromise', hue='bin_player_partner_diff', style='attack',
             palette=['salmon','gray', 'teal'])
plt.legend(bbox_to_anchor=(1.05, 1))

# Fig3 model comparison

In [ ]:
def read_model_df(mname, fname, folder = folder):
    if fname != '':
        fname = '_' + fname
    if folder == "conf":
        # p7 = pd.read_csv(f'../model_fits/{mname}_pilot7{fname}.csv')
        p7 = pd.read_csv(f'../../pilot7_rep/model_fits/{mname}{fname}.csv')
        p7['subID']= p7['subID'] + 400

        df = pd.concat([p7, pd.read_csv(f"../model_fits/{mname}_conf{fname}.csv")])
    else:
        df = pd.read_csv(f"../model_fits/{mname}{fname}.csv")
        
    return df

In [ ]:
# def mybic(mse, k, N = 178): #176 trials becuase we ignored the first trial for each predator
#     return k * np.log(N) + N * np.log(mse / N)

# def myaic(mse, k, N = 178):
#     return k * 2 + N * np.log(mse / N)

def mybic(nll, k, N = 176): #176 trials becuase we ignored the first trial for each predator
    return k * np.log(N) + 2 * nll

def myaic(nll, k, N = 176):
    return k * 2 +2 * nll

In [ ]:
m1_list = [
        # "peppgFull_econ_Theta_mse", 
        # "rollingAverage_lrdecay2_peppgFull_econ_ThetaGamma", 
        "lrdecay2_peppgFull_econ_ThetaGamma"
        # "valueppgFull_econ_Theta_mse", 
        # "valueppgFull_econ_ThetaGamma_mse"
        ]
m2_list = ["arbWeight_llh", "updateTheta_llh", "asIfIdv_llh"] #"deltaWeight",

all_params_df = pd.DataFrame()
for mymodel in m1_list:
    for mymodel2 in m2_list:
        # params_df = read_model_df(f"{mymodel}_{mymodel2}", '')
        # try:
        params_df = read_model_df(f"{mymodel}_{mymodel2}", '')
        params_df['model_name'] = mymodel2
        # params_df = params_df.rename({'nll':'mse'}, axis=1)
        k = 4 if "asIfIdv" not in mymodel2 else 3 
        params_df['bic'] = params_df['nll'].apply(lambda x: mybic(x, k))
        params_df['aic'] = params_df['nll'].apply(lambda x: myaic(x, k))
        all_params_df = pd.concat([all_params_df, params_df])
        # except:
        #     pass

print(all_params_df['model_name'].unique())
print(len(all_params_df['subID'].unique()))
# if min(all_params_df['subID'])==1:
#     all_params_df['subID'] = all_params_df['subID'] - 1
# print(f"{len(params_df.subID.unique())} subs with avg nll = {np.mean(params_df.loc[params_df.nll!=np.inf]['nll'])}")
all_params_df.head()




In [ ]:
baseline = all_params_df[all_params_df['model_name'] == 'arbWeight_llh'][['subID', 'bic']].rename(columns={'bic': 'bic_baseline'})

all_params_df = all_params_df.merge(baseline, on='subID')
all_params_df['delta_bic'] = all_params_df['bic'] - all_params_df['bic_baseline']  # positive = worse than baseline, negative = better
all_params_df

In [ ]:
from scipy.stats import wilcoxon

models = all_params_df[all_params_df['model_name'] != 'arbWeight_llh']['model_name'].unique()

for model in models:
    subset = all_params_df[all_params_df['model_name'] == model]['delta_bic']
    stat, p = wilcoxon(subset)
    n_better = (subset < 0).sum()
    print(f"{model}: mean ΔBIC={subset.mean():.2f}, W={stat:.2f}, p={p:.3f}, n_better_than_baseline={n_better}/{len(subset)}")

In [ ]:
#make sure the order matches!
model_names = ['Other-regarding\npreference\n', 'Social\ninfluence', 'As-if\nindividual']

In [ ]:
sns.set_context("talk")
fig, ax = plt.subplots(1, 1, figsize=(3.5, 5.5))
sns.barplot(data = all_params_df, x='model_name', y='bic', color='gold',
            # errorbar="se", 
            # s=3, alpha=0.8, 
            order = m2_list)
# sns.boxplot(data = all_params_df, x='model_name', y='bic', showfliers=False)
#set xticks
plt.xticks(np.arange(len(model_names)), model_names, rotation=45)
plt.xlabel('')
plt.ylabel('BIC', fontsize=17)
plt.ylim(bottom=500)

# #print significance
# print(ss.ttest_rel(all_params_df.query('model_name=="arbWeight"')['bic'],
#              all_params_df.query('model_name=="stopLearning"')['bic']))
# print(ss.ttest_rel(all_params_df.query('model_name=="updateTheta"')['bic'],
#              all_params_df.query('model_name=="asIfIdv"')['bic']))

#add annotation
# pairs=[(0, 1), (0, 2), (1, 2)]
pairs = [(m2_list[i], m2_list[j]) for i in range(len(m2_list)) for j in range(i)]
# pairs = [(m2_list[0], m2_list[1]), (m2_list[0], m2_list[2])]
annotator = Annotator(ax, pairs, data=all_params_df, x='model_name', y='bic', 
                      order=m2_list)
#set configurations and apply
annotator.configure(test='Wilcoxon', text_format='star', loc='inside',
                    line_height =0, text_offset=-1, hide_non_significant=False)#for style
annotator.apply_and_annotate()

# plt.ylim([0, 200])
# plt.tight_layout()
if save:
    plt.savefig(f'../figs/{folder}/bic_{folder}.png', 
                bbox_inches='tight', dpi=200)
    
# after plt.ylim(bottom=600)
kwargs = dict(transform=ax.transAxes, color='k', clip_on=False, linewidth=1.5)
ax.plot((-0.015, 0.015), (-0.02, 0.02), **kwargs)   # bottom-left slash
ax.plot((-0.015, 0.015), (-0.035, 0.005), **kwargs)  # double slash

In [ ]:
best_models = pd.merge(all_params_df, all_params_df.groupby('subID', as_index=False)['bic'].min())
model_of_interest = "arbWeight_llh"
# model_of_interest = "asIfIdv"
# model_of_interest = "updateTheta"

total_subjects = len(all_params_df['subID'].unique())
proportion = len(best_models.query('model_name==@model_of_interest')) / total_subjects

print(f"Proportion of subjects fit best by {model_of_interest}: {proportion:.2f}")


In [ ]:
def count_freq(sc, rc, mytype):
    #sc in [trials * k]
    #rc in [trial, 1]
    freq = []
    for i, real_c in enumerate(rc):
        #maybe smooth this mse?
        if mytype=='l2' or mytype=='mse':
            freq.append(np.sum(np.abs(sc[i,:] - real_c) **2))
        elif mytype=='l1':
            freq.append(np.sum(np.abs(sc[i,:] - real_c)))

        # freq.append(np.sum(sc[i,:] == real_c))
    return np.mean(freq) / sc.shape[1]


def compare_sim(sim_df_grp, sim_df_idv, m2_list, mytype='l2', plot=False):
    # models = df.model_name.unique()
    # subs = set.intersection(*df.groupby('model_name')['subID'].apply(set).values)
    subs  = sim_df_grp['subID'].unique()
    mse = np.zeros([len(subs), 2, len(m2_list)]) #num_subs * num_predators * num_models
    
    for i, sub in enumerate(subs): #loop over each subject
        for pt in [0, 1]: #loop over each predator
            # try:
            dist = 0
            # subset_idv = df_idv.query('subID==@sub and predatorType==@pt and trial>60')
            # subset_grp = df_group.query('subID==@sub and predatorType==@pt')
            # sub_resp = subset_grp.query('step_rt<8 and playerStep>0')['trial'].values
            subset_sim_grp = sim_df_grp.query('subID==@sub and predatorType==@pt and step_rt<8 and playerStep>0')
            subset_sim_idv = sim_df_idv.query('subID==@sub and predatorType==@pt and trial>60')
            # print(f"{sub}number of responses: {len(sub_resp)} and {len(subset_sim_grp)}")
            
            for j, m in enumerate(m2_list): #loop over each model      
                model_sim_data = subset_sim_grp.query('model_name==@m')
                sc = model_sim_data['sim_playerStep'].values
                try:
                    sc = [np.mean(i) for i in sc]
                except:
                    pass
                # print(sc)
                rc = model_sim_data['playerStep'].values
                # sc = np.reshape(sc, [len(rc), -1])
                # print(sc.shape)
                #ignore auto response
                # sc = sc[sub_resp - 1, :]
                # rc = rc[sub_resp - 1]
                # feq = count_freq(sc, rc, mytype)
                if mytype=='l2' or mytype=='mse':
                    feq = np.mean((sc - rc) ** 2)
                elif mytype=='l1':
                    feq = np.mean(np.abs(sc - rc))

                # sc = subset_sim_idv.query('model_name==@m')['sim_choice'].values
                # rc = subset_idv['choice'].values
                # dist2, ylab = get_dist(sc, rc, mytype)
                mse[i, pt, j] = feq
            # except:
            #     print(sub)
        
    #flatten with the predator dimension
    mse = np.nanmean(mse, axis=1) # shape: [sub, model]
    

    return mse

In [ ]:
new = pd.merge(
    df_idv.query('trial>60').groupby(['subID', 'predatorType'], as_index=False)['choice'].mean(),
        #   .agg(lambda x: list(x.mode())[0]),
    df_group[['playerStep', 'subID', 'predatorType', 'trial']],
    on=['subID', 'predatorType']
)
baseline_mse = new.groupby(['subID']).apply(lambda x: np.square(x['choice'] - x['playerStep']).mean())
plt.boxplot(baseline_mse, showfliers=False)
print(f"Baseline MSE: {baseline_mse.median():.2f} ± {baseline_mse.sem():.2f}")

In [ ]:
model_names_org = [f"{mymodel}_arbWeight_llh", f"{mymodel}_updateTheta_llh", f"{mymodel}_asIfIdv_llh"]
sim_type="full"

all_preds_grp = pd.DataFrame()
all_preds_idv = pd.DataFrame()
for mname in model_names_org:
    sim_g = read_model_df(mname, f'sim_group_{sim_type}')
    sim_g = pd.merge(sim_g, df_group[['subID', 'predatorType', 'trial', 'playerStep', 'step_rt']], how='inner')
    sim_g['sim_playerStep'] = sim_g['sim_playerStep'].apply(lambda x: eval(x))
    sim_i = read_model_df(mname, f'sim_idv_{sim_type}')
    sim_i = pd.merge(sim_i, df_idv[['subID', 'predatorType', 'trial', 'choice']], how='inner')
    # sim_i['sim_choice'] = sim_i['sim_choice'].apply(lambda x: eval(x))
    sim_g['model_name'] = mname.split("_")[-2]
    sim_i['model_name'] = mname.split("_")[-2]
    all_preds_grp = pd.concat([all_preds_grp, sim_g]) 
    all_preds_idv = pd.concat([all_preds_idv, sim_i])

m2_list = all_preds_grp.model_name.unique()
print(m2_list)
dist = compare_sim(all_preds_grp, all_preds_idv, m2_list, plot=False, mytype='l2')

In [ ]:
all_preds_grp = all_preds_grp.sort_values(by=['model_name', 'subID', 'predatorType', 'trial'])
all_preds_grp.head()

In [ ]:
# Convert the matrix into a DataFrame
df = pd.DataFrame(dist, columns=[i for i in range(dist.shape[1])])
# Melt the DataFrame to long format for seaborn compatibility
df_long = df.melt(var_name='X', value_name='Y')

# Create the swarm plot
fig, ax = plt.subplots(1, 1, figsize=(3.5, 5.5))
# sns.pointplot(x='X', y='Y', data=df_long, errorbar="se", color='gold')
sns.boxplot(x='X', y='Y', data=df_long, showfliers=False, color='gold', width=0.75)
# sns.violinplot(x='X', y='Y', data=df_long,color='gold')
#set x, y labels
plt.ylabel('simulated mse')
plt.xlabel('')
#rename models
plt.xticks(np.arange(len(model_names)), model_names, rotation=45)

#add annotation
pairs=[(0, 1), (0, 2), (1, 2)]
annotator = Annotator(ax, pairs, data=df_long, x='X', y='Y', order=[0, 1, 2])
#set configurations and apply
annotator.configure(test='t-test_paired', text_format='star', loc='inside',
                    line_height =0, text_offset=-0.1, hide_non_significant=False)#for style
annotator.apply_and_annotate()

# Add extra space at the top of the plot
# plt.ylim(1, 1.26) 

if save:
    plt.savefig(f'../figs/{folder}/simulated_mse_{folder}.png', 
                bbox_inches='tight', dpi=200)

In [ ]:
# model with clustering
# best_m = all_params_df.groupby(['subID'], as_index=False)['mse'].min()
# best_m = pd.merge(pd.merge(all_params_df, best_m), df_group[['subID', 'room']].drop_duplicates())
# c = pd.merge(collapsed_g[['room', 'cluster']], best_m)
c = pd.merge(collapsed_g[['room', 'cluster']], 
             pd.merge(all_params_df, df_group[['subID', 'room']].drop_duplicates()))
heat = c.groupby(['cluster', 'model_name'])['nll'].mean().reset_index()
heat_pivot = heat.pivot(
    index='model_name',
    columns='cluster',
    values='nll'
)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
sns.heatmap(
    heat_pivot,
    annot=True,
    fmt=".2f",
    cmap="viridis"  # or "coolwarm"
)

plt.xlabel("Cluster")
plt.ylabel("Model Name")
plt.title("NLL by Cluster and Model")
plt.show()

In [ ]:
#look at people who fit best by asIfIdv:

# s = best_models.query('model_name==@model_of_interest')['subID'].unique()
# rt_model = smf.mixedlm('log_step_rt ~ abs_self_pred_diff + playerStep', 
#                     data = rt_reg.query('subID in @s'),
#                     groups = rt_reg.query('subID in @s')['sub'],
#                     re_formula = '~ abs_self_pred_diff + playerStep'
#                 ).fit(reml=False, method="lbfgs", maxiter=2000)
# print(rt_model.summary())
# df_group.query('subID in @s and step_rt<8').groupby(['subID', 'predatorType'])['playerStep'].var()

In [ ]:
# fig, ax = plt.subplots(1, 1, figsize=(4.5, 5.5))
# sns.barplot(best_models.groupby(['model_name'])['subID'].count() / total_subjects, color='gold')
# #set xticks
# # plt.xticks(np.arange(len(model_names)), model_names, rotation=45)
# plt.xlabel('')
# plt.ylabel('percentage of subjects \nbest fit by each model')


### read best-fitting model

In [ ]:
# show simulations
def read_simulations(mname, folder, sim_type='partial'):
    sim_df_grp = read_model_df(mname, f'sim_group_{sim_type}')
    sim_df_idv = read_model_df(mname, f'sim_idv_{sim_type}')


    assert min(sim_df_grp['subID'])==0
    sim_df_grp = pd.merge(sim_df_grp.query('k==1'), df_group).sort_values(by=['subID', 'trial']).reset_index(drop=True)
    sim_g = sim_df_grp.groupby(['subID', 'predatorType'], as_index=False)[['playerStep', 'sim_playerStep']].mean() 

    sim_df_idv = pd.merge(sim_df_idv.query('k==1'), df_idv).sort_values(by=['subID', 'trial']).reset_index(drop=True)
    sim_i = sim_df_idv.groupby(['subID', 'predatorType'], as_index=False)[['choice', 'sim_choice']].mean()

    return sim_g, sim_i

In [ ]:
mymodel = "lrdecay2_peppgFull_econ_ThetaGamma"
mymodel2 = "arbWeight_llh"
mname = f"{mymodel}_{mymodel2}"

GENRULE, CHOICERULE = mymodel.split("_")[0:2]
print([GENRULE, CHOICERULE])

params_df = read_model_df(mname, '', folder)
# params_df['w'] = params_df['w'].apply(lambda x: x if x > -1 else -1)
print(f"{len(params_df)} subjects with avg mse {np.mean(params_df['nll'])}")
params_df.sort_values(by='subID').head()

In [ ]:
pair_params = pd.merge(df_group[['subID', 'room', 'playerID']].drop_duplicates(), params_df)
pair_params['partner_w'] = pair_params.groupby('room')['w'].transform(
    lambda x: x.iloc[::-1].values
)
pair_params = pd.merge(pair_params, get_ego_bias(df_group, groupby_cols=['subID']))
pair_params = pd.merge(pair_params, df_group.groupby(['subID'], as_index=False)['player_partner_diff'].mean())
pair_params.head()


In [ ]:
pair_params['player_partner_diff'] = np.abs(pair_params['player_partner_diff'])
smf.ols('w ~ ego_bias + player_partner_diff', data=pair_params).fit().summary()

In [ ]:
sns.lmplot(data = pair_params.query('playerID==1'), x='w', y='partner_w', 
        scatter_kws={'s':30, 'alpha':0.5})
r, p = ss.pearsonr(pair_params.query('playerID==1')['w'], 
        pair_params.query('playerID==1')['partner_w'])
p_txt = f'={round(p, 3)}' if p>0.001 else '<0.001'
plt.annotate(f"r={round(r, 2)}\np{p_txt}", xycoords='axes fraction', xy=(0.05, 0.82), fontsize=16)
if save:
    plt.savefig(f'../figs/{folder}/self_partner_w_corr_{folder}.png', 
                bbox_inches='tight', dpi=200)

In [ ]:


# controlling for player_partner_diff, ego_bias is not correlated with partner w
r, p = partial_corr_manual(pair_params, x='ego_bias', y='partner_w', covars=['player_partner_diff'], 
                            plot=True, save_fname='ego_corr_resid')
# print(f"r = {r:.3f}, p = {p:.3f}")

# controlling for player_partner_diff, ego_bias is still correlated with w
r, p = partial_corr_manual(pair_params, x='ego_bias', y='w', covars=['player_partner_diff'], 
                            plot=True, save_fname='ego_corr_resid')

# print(f"r = {r:.3f}, p = {p:.3f}")

In [ ]:
# check confidence
cfd = pd.merge(df_group.groupby(['subID'], as_index=False)[['confidence']].mean(), params_df)
# cfd = pd.merge(cfd, df_idv.groupby(['subID'], as_index=False)[['reward']].mean())
sns.lmplot(data = cfd, x='confidence', y='w', scatter_kws={'alpha':0.3, 's':25})
# ss.pearsonr(cfd['confidence'], cfd['w'])


In [ ]:
df_group['pred_error'] = abs(df_group['prediction'] - df_group['partnerStep'])
df_group['future_pred_error'] = (
    df_group.groupby(['subID', 'predatorType'])['pred_error'].shift(-1)
)

# d = df_group.query('selfBlame >= 0').dropna(
#     subset=['future_pred_error', 'attack', 'selfBlame', 'player_partner_diff']
# )
d = pd.merge(df_group.groupby(['subID'], as_index=False)[['pred_error', 'player_partner_diff']].mean(), params_df)
# pred_model = smf.mixedlm(
#     'pred_error ~ selfBlame *  attack',
#     data=d,
#     groups=d['subID'],
#     re_formula='~selfBlame * attack'
# ).fit()
pred_model = smf.ols('pred_error ~ w + player_partner_diff', data = d).fit()
pred_model.summary()

In [ ]:
d= df_group.query('selfBlame>-1 and trial<30').dropna(subset=['future_pred_error', 'selfBlame'])
pred_model = smf.mixedlm(
    'future_pred_error ~ player_partner_diff + selfBlame + attack',
    data=d,
    groups=d['subID'],
    re_formula='~ player_partner_diff + selfBlame + attack'
).fit()

pred_model.summary()

### model parameter distribution

In [ ]:
fig, axes = plt.subplots(1,4,figsize=(18,4))
axes[0].hist(params_df['alpha'])
axes[1].hist(params_df['gamma'])
axes[2].hist(params_df['theta'])
axes[3].hist(params_df['w'])
# axes[4].hist(params_df['delta'])
axes[0].set_title('learning rate \nalpha')
axes[1].set_title('value propagation param \ngamma')
axes[2].set_title('risk param \ntheta')
axes[3].set_title('social weight \nw')
plt.tight_layout()


In [ ]:
param_cols = ['alpha', 'gamma', 'theta', 'w']
params = params_df[param_cols]
params = params.rename({'alpha':r"$\alpha$", 
                        'gamma':r"$\gamma$",
                        'theta':r"$\theta$",
                        }, axis=1)

# Compute pairwise Pearson correlations
corr_matrix = params.corr(method='pearson')
corr_matrix = corr_matrix.round(2)
print(corr_matrix)

# Optional: visualize with seaborn heatmap
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Pairwise Correlation of Parameters")
plt.show()


# # Create pairwise scatterplots
# # Create a custom PairGrid to mask upper triangle
# g = sns.PairGrid(params, corner=True)  # corner=True shows only lower triangle

# # Add scatterplot and regression line to lower triangle
# g.map_lower(sns.regplot, scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'})

# # Optional: add histograms on diagonal
# g.map_diag(sns.histplot, kde=True)

In [ ]:
#box plot
plt.figure(figsize=(4, 9))
plt.ylim([-1.1, 1.6])
# t = params_df[['alpha_w', 'alpha_l', 'theta', 'w', 'subID']].melt(id_vars='subID')
t = params_df[['alpha', 'gamma', 'theta', 'w', 'subID']].melt(id_vars='subID')
sns.boxplot(data = t, x='variable', y='value', palette= sns.color_palette("Set2"), width=0.7)
plt.xticks(range(4), [r'$\alpha$', r'$\gamma$', r'$\theta$', r'$w$'])
plt.xlabel('model parameter')
plt.ylabel('value', labelpad=-1)

if save:
    plt.savefig(f'../figs/{folder}/params_dist_{folder}.png', 
                bbox_inches='tight', dpi=200)



# print summary statcistics
summary = params_df[['alpha', 'gamma', 'theta', 'w']].quantile([0.25, 0.5, 0.75]).T
summary.columns = ['25%', '50% (Median)', '75%']
# Step 2: Create table as plot
fig, ax = plt.subplots(figsize=(8, 2))
ax.axis('off')  # Hide axes

table = ax.table(
    cellText=summary.values,
    rowLabels=summary.index,
    colLabels=summary.columns,
    cellLoc='center',
    loc='center'
)

table.scale(1, 2)  # Scale table cells
table.auto_set_font_size(False)
table.set_fontsize(14)
print(summary)
if save:
    plt.savefig(f'../figs/{folder}/params_table_{folder}.png', 
                bbox_inches='tight', dpi=200)


In [ ]:
scatter_kws = {'alpha':0.25, 's':30, 'color':'black'}

def plot_param_correlation(d, xvar, yvar, xlabel, ylabel, xloc, yloc):
    d = pd.merge(params_df, d)

    plt.figure(figsize=(6, 5.5))
    sns.regplot(data=d, x=xvar, y=yvar, 
            scatter_kws = scatter_kws, line_kws={'color': 'black'},
            )
    r, p = ss.pearsonr(d[xvar], d[yvar])
    print([len(d), r, p])
    
    if p<0.001:
        p = '<0.001'
    else:
        p = f"={round(p, 3)}"
    if xvar =='theta':
        plt.xlim(left=xloc-0.1)
    if xvar=='alpha':
        plt.ylim(bottom=yloc-0.5)

    plt.ylabel(ylabel)
    plt.xlabel(xlabel)
    sns.despine()
    plt.text(
        xloc, yloc,
        f'r={r:.2f}\np{p}',
        fontsize=16
    )
    if save:
        plt.savefig(f'../figs/{folder}/supp_correlation_{xvar}_{folder}.png', 
            bbox_inches='tight', dpi=200)


# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
#is convergence correlated with w? larger weight on partner pref -> more convergence
d = df_group.groupby('subID', as_index=False
        )['player_partner_diff'].apply(lambda x: np.mean(np.absolute(x)))
plot_param_correlation(d, 'w', 'player_partner_diff', 'other regarding $w$', 'self-partner distance', 0.6, 6)

#higher prelec param -> underweighting of small probability -> risk-averse
#higher econ risk param ->risky
d = df_idv.query('trial>60').groupby(['subID'], as_index=False)['choice'].mean()
plot_param_correlation(d, 'theta', 'choice', 'risk parameter ' + r'$\theta$', 'individual foraging step', 0.1, 8)


#can different model predict different choice patterns towards different predators?
d = df_idv.query('trial>60').groupby(['subID', 'predatorType'], as_index=False)['choice'].mean()
d = pd.merge(d.query('predatorType==0'), d.query('predatorType==1'), on='subID')
d['choice_diff'] = d['choice_x'] - d['choice_y']
#correlate learning rates with differences in choice between predators
# sns.set_context('talk')
plot_param_correlation(d, 'alpha', 'choice_diff', 'learning rate '+r'$\alpha$', 'individual foraging step difference\n(low-threat - high-threat)', 0, -4)




In [ ]:
# alpha_t = alpha / sqrt(t) new
# alpha_t = alpha * (1 - t/60) before
# alpha_t = alpha

In [ ]:
#show recovery
# mymodel = "peppgFull_econ_ThetaGamma"
# mymodel2 = "arbWeight"

print(mname)

rec_df = read_model_df(mname, 'recovery')
m = pd.merge(params_df, rec_df, on=['subID'])
# # #if fit by sub:
# rec_df['subID'] = rec_df['subID']-1
# rec_df.head()
m.head()

sns.set_context('talk')
m['gamma2_x'] = m['gamma_x'] * m['alpha_x']
m['gamma2_y'] = m['gamma_y'] * m['alpha_y']
print(len(m))
col_names = {'alpha':r'learning rate $\alpha$',
             'theta': r'risk $\theta$',
             'gamma': r'discount $\gamma$',
             'w':r'weight $w$'}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, col in enumerate(['alpha', 'theta', 'gamma', 'w']):
    sns.regplot(data=m, x=col+'_x', y=col+'_y', ax=axes[i], scatter_kws={'s':30, 'alpha':0.2})
    # axes[i].plot([axes[i].get_xlim()[0], axes[i].get_xlim()[1]], [axes[i].get_ylim()[0], axes[i].get_ylim()[1]], 
    #              ls='--', lw=1.5)
    #add title
    # col = re.sub(r'\d+', '', col)
    # axes[i].set_title(f"$\{col}$" if col!='w' else f"${col}$")
    axes[i].set_title(col_names[col])
    axes[i].set_xlabel('fitted')
    axes[i].set_ylabel('recovered')
    print(ss.pearsonr(m[col+'_x'], m[col+'_y']))

    
plt.tight_layout()
if save:
    plt.savefig(f'../figs/{folder}/supp_recovery_{mymodel}_{mymodel2}_{folder}.png', 
                bbox_inches='tight', dpi=200)

In [ ]:
def read_simulations(mname, folder, sim_type='partial'):
    sim_df_grp = read_model_df(mname, f'sim_group_{sim_type}')
    sim_df_idv = read_model_df(mname, f'sim_idv_{sim_type}')
    sim_df_grp['sim_playerStep'] = sim_df_grp['sim_playerStep'].apply(lambda x: np.mean(eval(x)))
    sim_df_idv['sim_choice'] = sim_df_idv['sim_choices'].apply(lambda x: np.mean(eval(x)))

    assert min(sim_df_grp['subID'])==0
    sim_df_grp = pd.merge(sim_df_grp.query('k==1'), df_group).sort_values(by=['subID', 'trial']).reset_index(drop=True)
    sim_g = sim_df_grp.groupby(['subID', 'predatorType'], as_index=False)[['playerStep', 'sim_playerStep']].mean() 

    sim_df_idv = pd.merge(sim_df_idv.query('k==1'), df_idv).sort_values(by=['subID', 'trial']).reset_index(drop=True)
    sim_i = sim_df_idv.groupby(['subID', 'predatorType'], as_index=False)[['choice', 'sim_choice']].mean()

    return sim_g, sim_i

In [ ]:
sim_g, sim_i = read_simulations(mname, folder, 'full')

In [ ]:
print(len(sim_g.subID.unique()))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 9), sharex=True, sharey=True)
colors=sns.color_palette()
for i in [0, 1]: #fore each predator
    axes[0, i].plot([1, 9], [1, 9], ls='--', color='black', alpha=0.5)
    axes[1, i].plot([1, 9], [1, 9], ls='--', color='black', alpha=0.5)
    #idv
    sns.scatterplot(x=sim_i.query('predatorType==@i')['choice'], 
                    y=sim_i.query('predatorType==@i')['sim_choice'], 
                    ax=axes[0, i], color=colors[i], s=50)
    #group
    sns.scatterplot(x=sim_g.query('predatorType==@i')['playerStep'], 
                    y=sim_g.query('predatorType==@i')['sim_playerStep'], 
                    ax=axes[1, i], color=colors[i], s=50)

    axes[0, i].set_facecolor((255/255, 215/255, 0/255, 0.15))
    axes[1, i].set_facecolor((147/255, 112/255, 219/255, 0.15))
    axes[i, 1].tick_params(left=False)
    axes[0, i].tick_params(bottom=False)
plt.subplots_adjust(wspace=0, hspace=0)
# axes[0, 0].set_title('safe predator')
# axes[0, 1].set_title('dangerous predator')

axes[0, 0].set_ylabel('simulated individual\n foraging choice', fontsize=22)
axes[1, 0].set_ylabel('simulated dyad\n foraging choice', fontsize=22)

axes[1, 0].set_xlabel("real choice\nlow-threat predator", fontsize=22)
axes[1, 1].set_xlabel("real choice\nhigh-threat predator", fontsize=22)
axes[1, 0].set_xticks([2, 4, 6, 8])
axes[1, 1].set_xticks([2, 4, 6, 8])
if save: 
    plt.savefig(f'../figs/{folder}/simulations_{folder}.png',
            bbox_inches='tight', dpi=200)

In [ ]:
# mname_alt = f"{mymodel}_asIfIdv"
# mname_alt = f"{mymodel}_updateTheta"
mname_alt = f"{mymodel}_arbWeight_llh"

# replicate regression
sim_df_grp = read_model_df(mname_alt,'sim_group_full')
sim_df_idv = read_model_df(mname_alt,'sim_idv_full')
sim_df_idv = pd.merge(sim_df_idv, df_idv[['subID', 'trial', 'predatorType']])

sim_df_grp = pd.merge(sim_df_grp, df_group, how='right').sort_values(by=['subID', 'trial']).reset_index(drop=True)
sim_df_grp["sim_playerStep"] = sim_df_grp["sim_playerStep"].fillna(sim_df_grp["playerStep"])
sim_df_grp["sim_attack"] = sim_df_grp["sim_attack"].fillna(sim_df_grp["attack"])
sim_df_grp["k"] = sim_df_grp["k"].fillna(1)
sim_df_grp['sim_player_change'] = sim_df_grp.groupby(['subID', 'predatorType'])['sim_playerStep'].shift(-1) - sim_df_grp['sim_playerStep']
sim_df_grp['sim_player_partner_diff'] = sim_df_grp['sim_playerStep'] - sim_df_grp['partnerStep']
sim_df_grp['sim_attack'] = sim_df_grp['sim_attack'].astype(bool)
sim_df_grp['sim_avgMoney'] = sim_df_grp.apply(lambda row: -10 if row['sim_attack']==1 else np.ceil((row['sim_playerStep'] + row['partnerStep']) / 2) **2, axis=1)

In [ ]:
d = pd.merge(sim_df_grp.groupby('subID', as_index=False)['sim_avgMoney'].mean(),
         df_idv.query('trial>60').groupby('subID', as_index=False)['reward'].mean())
sns.barplot(data = d.melt(id_vars='subID'), x='variable', y='value')

In [ ]:
# df_group = df_group.sort_values(by=['subID', 'trial'])
# df_group['player_change'] = df_group.groupby(['subID', 'predatorType'])['playerStep'].shift(-1) - df_group['playerStep']
def plot_stepchange_heatmap(df_group, sim='sim_'):
    pivot_df = df_group.query('partnerStep>0 and trial<30 and playerStep>0').groupby(['subID', 'partnerStep', f'{sim}playerStep'], as_index=False)[f'{sim}player_change'].mean()
    pivot_df = pivot_df.pivot_table(
        index="partnerStep",            # y-axis
        columns=f'{sim}playerStep',       # x-axis
        values=f'{sim}player_change',     # cell values
        aggfunc="mean",                 # or "median", "sum", etc.
        fill_value=np.nan,              # or 0 if you prefer
    )

    plt.figure(figsize=(6,5))
    sns.heatmap(pivot_df, annot=False, cmap="coolwarm", 
                vmin=-3.5, vmax=3.5, 
                cbar_kws={'label': f'{sim}player change'})
    plt.xlabel("playerStep")
    plt.ylabel("partnerStep")
    # plt.title("Heatmap of player step change")
    plt.tight_layout()
    plt.show()

In [ ]:
# plot_stepchange_heatmap(df_group, sim='')

In [ ]:
# can you get the same regression result with simulated data?

sim_df_grp['sim_ms_playerStep'] = sim_df_grp['sim_playerStep'] - 4.5
df_change_sim_reg = sim_df_grp.query('trial<30').dropna() #remove first and last trial and auto choices

print(len(df_change_sim_reg['subID'].unique()))
change_model_sim = smf.mixedlm('sim_player_change ~ sim_ms_playerStep + sim_player_partner_diff * sim_attack', 
                    data = df_change_sim_reg,
                    groups = df_change_sim_reg['subID'],
                    re_formula = '~ sim_ms_playerStep + sim_player_partner_diff * sim_attack'
                ).fit(reml=False, maxiter=2000)
print(change_model_sim.summary())

# with open(f'../figs/{folder}/change_reg_summary_{mname_alt}_{folder}.txt', "w") as f:
#     f.write(change_model_sim.summary().as_text())

In [ ]:
sns.set_context('talk')
compromise_colors = ['firebrick', 'steelblue'] #high, los

#use theta, w
t = df_group.query('step_rt<8 and selfBlame>-1').groupby(['subID', 'room', 'attack'], as_index=False)['selfBlame'].mean()
t = pd.merge(t.query('attack==False'), t.query('attack==True'), on=['subID', 'room'])
t['blame_diff'] = t['selfBlame_x'] - t['selfBlame_y'] #attack_false - attack_win
merged = pd.merge(params_df, t)
print(f"{len(merged)} subjects")

# get riskyiness within dyad through theta
merged = pd.merge(merged.groupby(['room'], as_index=False)['theta'].mean().rename({'theta':'mean_theta'}, axis=1), 
                  merged, how='right')
merged['risky_wpair'] = merged.apply(lambda x:'higher' if x['theta']>x['mean_theta'] else 'lower', axis=1)


# if correlation
sns.lmplot(data = merged, x='w', y='blame_diff', scatter_kws={"s": 20, 'alpha':0.5})
plt.ylabel('$blame_{win} - blame_{lose}$')
r, p = ss.pearsonr(merged['w'], merged['blame_diff'])
print(f"Correlation between w and blame_diff: r={r:.2f}, p={p:.3f}")
p = f"={round(p, 3)}" if p>=0.001 else "<0.001"
plt.annotate(f'r={round(r, 2)}\np{p}', xy=(0.75, 0.9), xycoords='axes fraction', fontsize=16)
plt.ylabel('Egocentric bias')
plt.xlabel('$w$', labelpad=10)
plt.tight_layout()
if save:
    plt.savefig(f'../figs/{folder}/correlation_w_bias_{folder}.png', 
            bbox_inches='tight', dpi=200)



# if binarize
fig, ax = plt.subplots(1, 1, figsize=(4, 5))
median_w = np.median(params_df['w'])
print(f"median w: {median_w}")
merged['compromise'] = merged['w'].apply(lambda x: 'high' if x>= median_w else 'low')
sns.barplot(data=merged, x='compromise', y='blame_diff', 
            palette=compromise_colors, order=['high', 'low'], 
            alpha=0.7
            )
sns.swarmplot(data=merged, x='compromise', y='blame_diff', 
            palette=compromise_colors, order=['high', 'low'], 
            alpha=0.5, s=3
            )
plt.ylabel('Egocentric bias')
plt.xlabel('Compromise', labelpad=10)
# plt.ylabel('$mean (credit - blame)$')
# plt.ylim([0, 0.21])
# ax.set_ylim([-0.35, 0.75])
ax.set_yticklabels(['{:.0f}%'.format(x*100) for x in ax.get_yticks()])



# Perform a t-test to compare the two groups
group_low = merged[merged['compromise'] == 'high']['blame_diff']
group_high = merged[merged['compromise'] == 'low']['blame_diff']
t_stat, p_value = ss.ttest_ind(group_low, group_high)
print(f"t = {t_stat}, p = {p_value}")
# Annotate the plot with significance if p-value < 0.05
sig = get_sig(p_value)
plt.text(0.5, 0.19, sig, ha='center', va='bottom')
plt.plot([0, 0, 1, 1], [0.19, 0.19, 0.19, 0.19], lw=1.5, color='black')

plt.tight_layout()
# if save:
#     plt.savefig(f'../figs/{folder}/correlation_w_bias_{folder}.png', 
#             bbox_inches='tight', dpi=200)


In [ ]:
merged = pd.merge(merged, df_group.groupby(['subID'], as_index=False)['player_partner_diff'].mean())
merged['player_partner_diff'] = np.abs(merged['player_partner_diff'])
merged

In [ ]:
from sklearn.linear_model import LinearRegression

def residualize(y, X, df):
    model = LinearRegression().fit(df[[X]], df[y])
    return df[y] - model.predict(df[[X]])

merged['ego_bias_resid'] = residualize('blame_diff', 'player_partner_diff', merged)
merged['w_resid'] = residualize('w', 'player_partner_diff', merged)

# then correlate residuals
r, p = pearsonr(merged['ego_bias_resid'], merged['w_resid'])

sns.lmplot(data = merged, x='ego_bias_resid', y='w_resid', scatter_kws={"s": 20, 'alpha':0.5})
p = f"={round(p, 3)}" if p>=0.001 else "<0.001"
plt.annotate(f'r={round(r, 2)}\np{p}', xy=(0.75, 0.9), xycoords='axes fraction', fontsize=16)

In [ ]:
#compare w between risky and risk-averse
fig, ax= plt.subplots(1, 1, figsize=(4, 5))
merged['theta_wpair'] = merged.apply(lambda x: 'higher' if x['theta'] >x['mean_theta'] else 'lower', axis=1)
sns.swarmplot(data=merged, x='risky_wpair', y='w',
            alpha=0.7, s=3.5, palette=risk_colors)
sns.barplot(data=merged, x='risky_wpair', y='w', 
            alpha=0.5, palette=risk_colors)
pairs = [("higher", "lower")]
annotator = Annotator(ax, pairs, data=merged,  x='risky_wpair', y='w')
#set configurations and apply
annotator.configure(test='t-test_ind', hide_non_significant=False,
                    # comparisons_correction="bonferroni", 
                    text_format='star', loc='inside', 
                    line_height =0, text_offset=-1)#for style
annotator.apply_and_annotate()

           
#risky people lose money from compromising?
#people who compromised improve performance?
#y_axis: improve, x: cluster, hue='risky
# sns.catplot(data = df, y='grp_idv_step_diff', x='cluster_category', hue='risky_wpair', kind='bar')
# plt.xticks(rotation=45)
# ss.pearsonr(df.query('risky_wpair=="higher"')['grp_idv_step_diff'], df.query('risky_wpair=="higher"')['grp_idv_reward_diff'])
# on average, people performed better in groups
# d = pd.merge(merged[['subID', 'compromise', 'w', 'theta_wpair', 'theta']], 
#              pd.merge(df_group[['subID', 'room']].drop_duplicates(), group_stat))
# d = d.groupby(['subID', 'risky_wpair', 'compromise'], as_index=False)['grp_idv_reward_diff'].mean()
# sns.swarmplot(data = d, y='grp_idv_reward_diff', hue='risky_wpair', x='compromise', palette=['salmon', 'teal'],
#             dodge=True)


# fig, ax = plt.subplots(1,1,figsize=(4, 5))
# # sns.lmplot(data = d, y='reward_inc', x='w', hue='risky_wpair',
# #             scatter_kws={'s':10, 'alpha':0.5}, )
# # plt.xlabel('w')
# # plt.ylabel('$\Delta$ reward')
# # print(ss.pearsonr(d['reward_inc'], d['w']))
# d = pd.merge(params_df, group_stat.groupby(['subID'], as_index=False)[['reward_inc']].mean())
# d['compromise'] = d['w'].apply(lambda x: 'high' if x>=np.median(d['w']) else 'low')
# d['reward_inc']= d['reward_inc'] * 60
# d = pd.merge(d, merged[['subID', 'theta_wpair']])
# sns.swarmplot(data = d, y='reward_inc', x='compromise', alpha=0.4, s=4, palette=compromise_colors)
# sns.barplot(data = d, y='reward_inc', x='compromise', alpha=0.6, palette=compromise_colors)
# plt.ylabel(r'$reward_{dyad}$ - $reward_{idv}$')
# r, p = ss.ttest_ind(d.query('compromise=="high"')['reward_inc'], d.query('compromise=="low"')['reward_inc'])
# print(r, p)
# sig = get_sig(p)
# plt.text(0.5, 250, sig, ha='center', va='bottom', fontsize=24)
# plt.plot([0, 0, 1, 1], [250, 250, 250, 250], lw=1.5, color='black')
# if save:
#     plt.savefig(f'../figs/{folder}/reward_by_compromise{folder}.png', 
#             bbox_inches='tight', dpi=200)

# print(ss.pearsonr(d.query('risky_wpair=="risk-prone"')['reward'], 
#                   d.query('risky_wpair=="risk-prone"')['w']))
# print(ss.pearsonr(d.query('risky_wpair=="risk-averse"')['reward'], 
#                   d.query('risky_wpair=="risk-averse"')['w']))`
# reward ~ w * risky_wpair


# plt.legend(fontsize=13, title='within dyad', title_fontsize=14)
# plt.xticks([0, 1], ['no-compromise', 'compromise'])


# plt.axhline(0, color='black', lw=0.75, ls='--')


# **One-sample t-tests against 0**
# p_values = {}
# for (comp, risky), group in d.groupby(['compromise', 'risky_wpair']):
#     if len(group) > 1:  # Ensure at least two samples for the test
#         stat, p_val = ss.ttest_1samp(group['reward_inc'], 0)
#         p_values[(comp, risky)] = p_val
#     else:
#         p_values[(comp, risky)] = np.nan  # Avoid errors with single observations

# # **Manual annotation of p-values**
# for (comp, risky), p_val in p_values.items():
#     if not np.isnan(p_val):
#         x_pos = 0 if comp=="higher" else 1  # Convert boolean to integer (0 or 1 for x-axis placement)
#         y_pos = d[(d['compromise'] == comp) & (d['risky_wpair'] == risky)]['reward_inc'].max() + 1  # Position above max value
#         plt.text(x_pos, y_pos, f"p={p_val:.3f}", ha='center', fontsize=12, color='black')

# plt.show()

#barplot for simplicity
# sns.catplot(data=df_perf, x='compromise', y='grp_idv_reward_diff', kind='bar')
# df.groupby(['compromise', 'risky_wpair'])['sub'].count()


In [ ]:
#or can you plot betas?

## run one regression, and plot betas for each compromise level
df_filtered = df_group.query('selfBlame>-1').copy()
#drop na value for regression
df_filtered = df_filtered[['selfBlame', 'player_partner_diff', 'outcome', 'subID', 'trial', 'predatorType']].dropna()

model_baseline = smf.mixedlm("selfBlame ~ player_partner_diff * outcome", data=df_filtered,
                    groups=df_filtered["subID"],
                    re_formula="~ player_partner_diff * outcome")
result_baseline = model_baseline.fit()

print(result_baseline.summary())

In [ ]:
# separate by compromise
median_w = np.median(params_df['w'])

high_w = params_df.query('w>@median_w')['subID'].unique()
d = df_group.query('selfBlame>-1').groupby(['subID', 'attack', 'player_partner_diff'], 
                                            as_index=False)['selfBlame'].mean()
d['compromise'] = d['subID'].apply(lambda x: "high" if x in high_w else "low")
d = pd.merge(d, params_df[['subID', 'w']])

d['outcome'] = d['attack'].apply(lambda x: 'harvest' if x==False else 'captured').astype('category')
d['outcome'] = d['outcome'].cat.reorder_categories(['harvest', 'captured'], ordered=True)

#plot
plt.figure(figsize=(6, 5))
plt.axhline(0.5, ls='--', color='gray', lw=0.8)
plt.axvline(0, ls='--', color='gray', lw=0.8)
ax = sns.lineplot(
    data=d.query('player_partner_diff >= -8 and player_partner_diff<=8'), 
    x='player_partner_diff', y='selfBlame', 
    hue='outcome', 
    style='compromise', 
    palette=outcome_colors,
    alpha=0.75,
    lw = 1.5,
    style_order=['high', 'low'],
    legend='brief'  # include just one legend entry per style/hue combo
)

# Remove hue entries manually
handles, labels = ax.get_legend_handles_labels()

# Keep only unique 'style' labels (assuming 'compromise' has unique line styles)
# This assumes style names (like 'compromise' values) are in `labels`
style_labels = d['compromise'].unique().tolist()
style_handles = [h for h, l in zip(handles, labels) if l in style_labels]

#set x/y label and legend
ax.legend(style_handles, style_labels, title='compromise')
ax.set_xlabel('$S_{player} - S_{partner}$')
ax.set_ylabel('credit / blame attributed to oneself', labelpad=0.5)

if save:
    plt.savefig(f'../figs/{folder}/bias_by_compromise_{folder}.png', 
            bbox_inches='tight', dpi=200)

In [ ]:
# d['outcome'] = d['attack'].apply(lambda x: 'lose' if x ==True else 'win')
if np.min(d['selfBlame']) ==0:
    d['selfBlame'] = d['selfBlame'] - 0.5
# d['compromise2'] = d['compromise'].apply(lambda x: 1 if x=='low' else 0)
# model = smf.mixedlm(formula='selfBlame ~ outcome * compromise + player_partner_diff * compromise + player_partner_diff * outcome', 
#                     data=d, groups = d['subID'],
#                     re_formula="~ outcome + player_partner_diff + player_partner_diff * outcome"
#                     )
d = d.drop({'compromise'}, axis=1).rename({'w':'compromise'}, axis=1)
# d['selfBlame'] =d.apply(lambda row: 1-row['selfBlame'] if row['attack']==True else row['selfBlame'], axis=1) #rescale to -1 to 1
model = smf.mixedlm(formula='selfBlame ~ outcome * compromise + player_partner_diff * compromise + player_partner_diff * outcome', 
                    data=d, groups = d['subID'],
                    re_formula="~ outcome * compromise + player_partner_diff * compromise + player_partner_diff * outcome"
                    )
result = model.fit()
# model = smf.mixedlm(formula='selfBlame ~ player_partner_diff * compromise* outcome', 
#                     data=d, groups = d['subID'],
#                     re_formula="~ player_partner_diff * compromise * outcome"
#                     )
# result = model.fit()
print(result.summary())

In [ ]:


# Extract fixed effects and confidence intervals
fe_params = result.fe_params
ci = result.conf_int()
ci.columns = ['lower', 'upper']

# Combine into a single DataFrame
ci['estimate'] = fe_params
ci = ci.reset_index().rename(columns={'index': 'term'})

# Sort terms (optional, for cleaner plotting)
ci = ci.sort_values(by='term', ascending=False).dropna()
label_map = {
    'Intercept': 'Intercept',
    'outcome[T.captured]': 'Outcome [captured]',
    'compromise': 'Compromise',
    'player_partner_diff': 'Self-partner distance',
    'outcome[T.captured]:compromise': 'Outcome [captured] \nx Compromise',
    'player_partner_diff:compromise': 'Self-partner distance \nx Compromise',
    'player_partner_diff:outcome[T.captured]': 'Self-partner distance \nx Outcome [captured]',
}
ci['term_pretty'] = ci['term'].map(label_map)
# Plot
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
plt.errorbar(
    x=ci['estimate'],
    y=ci['term_pretty'],
    xerr=[ci['estimate'] - ci['lower'], ci['upper'] - ci['estimate']],
    fmt='o',
    color='black',
    ecolor='gray',
    capsize=4
)
plt.axvline(x=0, color='gray', linestyle='--')
plt.xlabel('fixed effect estimates')
# plt.ylabel('Term')
for tick_label in ax.get_yticklabels():
    if tick_label.get_text() in ['Outcome [captured] \nx Compromise','Compromise', 'Self-partner distance \nx Compromise']:
        tick_label.set_color('red')

plt.tick_params(labelsize=14) #axis='y', 
# plt.title('Fixed Effects with 95% Confidence Intervals')
plt.tight_layout()
if save:
    plt.savefig(f'../figs/{folder}/blame_betas_compromise_{folder}.png', 
            bbox_inches='tight', dpi=200)
plt.show()


In [ ]:
# Convert random effects to DataFrame
random_effects = result.random_effects
betas_df = pd.DataFrame.from_dict(random_effects, orient='index')
betas_df.reset_index(inplace=True)
betas_df.rename(columns={'index': 'subID'}, inplace=True)

# Add fixed effects to get subject-level total effects
for col in result.fe_params.index:
    if col in betas_df.columns:
        betas_df[col] = betas_df[col] + result.fe_params[col]

betas_df['intercept'] = betas_df['Group'] + result.fe_params['Intercept']

betas_df['group'] = betas_df['subID'].apply(lambda x: "high" if x in high_w else "low")

In [ ]:

# #ifcontinuous
# plt.figure(figsize=(6, 4.5))
# d = group_stat.copy()
# d['step_inc_directed'] = np.round(d['step_inc_directed']*2 )//2
# sns.lineplot(data = d, x='step_inc_directed', y='reward_inc', hue='risky_wpair', palette=risk_palette)
# plt.axhline(0, ls='--', color='black', lw=1)
# plt.axvline(0, ls='--', color='black', lw=1)
# plt.xlabel('compromise')
# plt.legend(loc='lower center', fontsize=14, title_fontsize=15, title='riskiness within dyad')
# plt.ylabel(r'$reward_{dyad}$ - $reward_{idv}$')
# # plt.xlim([-7, 7])

In [ ]:
# control for reward by time
# no increase in reward in the later idv trials
# d = df_idv.query('trial>60').copy()

# d['cum_avg_reward'] = d.groupby(['subID', 'predatorType'])['reward'].cumsum()
# d['cum_avg_reward'] = d['cum_avg_reward'] / d['num_encounter']
# d['trial'] = d['num_encounter'] - 30
# df_group['cum_avg_reward'] = df_group.groupby(['subID', 'predatorType'])['jointMoney'].cumsum()
# df_group['cum_avg_reward'] = df_group['cum_avg_reward'] / 2/  df_group['trial']



plt.figure(figsize=(6, 4.5))
df_group['avgMoney'] = df_group['jointMoney'] / 2
sns.lineplot(data = df_idv.query('num_encounter<60 and num_encounter>30'), x='num_encounter', y='reward', 
             hue='predator', hue_order=['low-threat', 'high-threat'], legend=False)
sns.lineplot(data = df_group, x='trial', y='avgMoney', hue='predator', hue_order=['low-threat', 'high-threat'])
plt.axvline(x=30, color='black', ls='-')
plt.gca().invert_xaxis()
plt.xticks([])
plt.legend(fontsize=14, loc='center left', title='predator', title_fontsize=16)
plt.xlabel('individual trial                dyad trial')
# plt.ylim([0, 26])
if save:
    plt.savefig(f'../figs/{folder}/reward_by_trial_{folder}.png', 
            bbox_inches='tight', dpi=200)

#no obv increase in individual
smf.ols("reward ~ num_encounter * C(predatorType)",
                    data=df_idv.query('num_encounter<60 and num_encounter>30'),
                    ).fit().summary()
# smf.ols("avgMoney ~ trial * C(predatorType)",
#                     data=df_group,
#                     ).fit().summary()

In [ ]:

ss.ttest_1samp(group_stat.groupby('subID')['reward_inc'].mean(), 0)

In [ ]:
# 

# Optimality

In [ ]:
#plot optimal
optimal = []
predators=['low-threat', 'high-threat']
for pt in [0, 1]:
    attack_prob = df_idv.query('predatorType==@pt and choice>0').groupby(['choice'], as_index=False)['currProb'].mean()
    values = (1 - attack_prob['currProb'].values) * (attack_prob['choice'] ** 2) - attack_prob['currProb'].values * 10
    plt.plot(values, c=sns.color_palette()[pt], label=predators[pt])
    opt_step = np.argmax(values)
    plt.axvline(opt_step, ls='--', color=sns.color_palette()[pt], lw=1)
    optimal.append(opt_step + 1) #turn to 1-indexed

plt.xticks(np.arange(9), np.arange(9)+1)
plt.xlabel('step')
plt.ylabel('expected reward')
plt.legend(title='predator')


# compromise by optimality
print(f"predator 0 : {optimal[0]}, predator 1 : {optimal[1]}")
group_stat['deviation_from_optimal'] = group_stat.apply(lambda row: row['individual'] - optimal[0] if row['predator']==0 else row['individual'] - optimal[1], axis=1)

In [ ]:
#plot attack rate
optimal_att = []
predators=['low-threat', 'high-threat']
for pt in [0, 1]:
    attack_prob = df_idv.query('predatorType==@pt and choice>0').groupby(['choice'])['currProb'].mean()
    attack_prob[attack_prob>1] = 1
    
    optimal_att.append(attack_prob[optimal[pt]])
    plt.plot(attack_prob, c=sns.color_palette()[pt], label=predators[pt])
    
# plt.xticks(np.arange(9), np.arange(9)+1)
plt.xlabel('step')
plt.ylabel('attack probability')
plt.legend(title='predator')
print(optimal_att)


In [ ]:
sns.lmplot(data = group_stat, x='deviation_from_optimal', y='step_inc_towards_partner', hue='risky_wpair',
           scatter_kws={'s':12, 'alpha':0.25}, palette=risk_palette)
plt.axhline(0, ls='--', lw=0.8, color='black')
plt.axvline(0, ls='--', lw=0.8, color='black')
plt.ylabel('compromise')

In [ ]:
# sns.lmplot(data = group_stat, x='deviation_from_optimal', y='step_inc_directed', hue='risky_wpair',
#            scatter_kws={'s':15, 'alpha':0.2}, palette=risk_palette)

d = group_stat.copy()
d['deviation_from_optimal'] = np.abs(d['deviation_from_optimal'])
sns.lmplot(data = d, x='deviation_from_optimal', y='step_inc_towards_partner', hue='predator',
           scatter_kws={'s':12, 'alpha':0.2}, legend=False)
plt.ylabel('compromise')
plt.xlabel('deviation from optimal')
print("high-threat: ")
print(ss.pearsonr(d.query('predatorType==1')['deviation_from_optimal'], 
                  d.query('predatorType==1')['step_inc_towards_partner']))
print("low-threat: ")
print(ss.pearsonr(d.query('predatorType==0')['deviation_from_optimal'], 
                  d.query('predatorType==0')['step_inc_towards_partner']))

r, p= ss.pearsonr(d['deviation_from_optimal'], d['step_inc_towards_partner'])
p = f"p = {round(p, 3)}" if p>0.001 else "p < 0.001"
plt.xlim(right=4.5)
plt.ylim(top = 8)
plt.annotate(f"r = {round(r, 2)}\n{p}", xy=(0.1, 6), fontsize=15)


if save:
    plt.savefig(f'../figs/{folder}/compromise_optimal_{folder}.png', 
                bbox_inches='tight', dpi=200)

In [ ]:


# Compute means
df_grp_mean = (
    df_group.groupby(['subID', 'predator'], as_index=False)['attack']
    .mean()
    .assign(condition='Group')
)

df_idv_mean = (
    df_idv.groupby(['subID', 'predator'], as_index=False)['attack']
    .mean()
    .assign(condition='Individual')
)

# Combine
df_all = pd.concat([df_idv_mean, df_grp_mean], ignore_index=True)
df_all = pd.merge(df_all, group_stat[['subID', 'predator', 'risky_wpair']], how='left')
# Plot
plt.figure(figsize=(6,4))
sns.pointplot(
    data=df_all.query('risky_wpair=="risk-prone"'),
    hue='predator',
    y='attack',
    x='condition',
    legend=False,
    dodge=0.2
    # ci='ci'   # or "se" / None, depending on what you want
)

sns.pointplot(
    data=df_all.query('risky_wpair=="risk-averse"'),
    hue='predator',
    y='attack',
    x='condition',
    marker = '^',
    ls=':',
    legend=False,
    dodge=0.2
    # ci='ci'   # or "se" / None, depending on what you want
)
plt.ylabel("Mean attack rate")
plt.ylim(top=0.55)
# plt.xlabel("Predator type")
plt.title("Attack rates in Group vs Individual conditions")
legend_handles = [
    Line2D([0], [0], color='black', ls='--', marker='^', label='risk-averse'),
    Line2D([0], [0], color='black', ls='-', marker='o', label='risk-prone')
]
plt.legend(handles = legend_handles, title='riskiness within dyad', title_fontsize=16, fontsize=15)



In [ ]:
# sns.barplot(data = df_group.query('(predatorType==0 and finalStep==8) or (predatorType==1 and finalStep==5)'), x='predatorType', y='attack')
# sns.barplot(data = df_group.groupby(['subID', 'predatorType'], as_index=False)['attack'].mean(), x='predatorType', y='attack')
# sns.barplot(data = df_idv.groupby(['subID', 'predatorType'], as_index=False)['attack'].mean(), x='predatorType', y='attack')

# 1. Prepare your three datasets with a new column "source"
d1 = df_group.query('(predatorType==0 and finalStep==8) or (predatorType==1 and finalStep==5)').copy()
d1['source'] = "optimal"

d2 = df_group.groupby(['subID', 'predator'], as_index=False)['attack'].mean()
d2['source'] = "dyad"

d3 = df_idv.groupby(['subID', 'predator'], as_index=False)['attack'].mean()
d3['source'] = "individual"

# 2. Combine
df_all = pd.concat([d1[['predator','attack','source']],
                    d3[['predator','attack','source']],
                    d2[['predator','attack','source']]])

# 3. Plot
plt.figure(figsize=(5,5))
sns.pointplot(data=df_all, hue='predator', y='attack', x='source', hue_order=['low-threat', 'high-threat'])
plt.ylabel("frequency of captures")
plt.xlabel("")
# plt.title("Attack by predator type across datasets")
plt.legend(title="predator")
# plt.axhline(optimal_att[0], ls='--')
# plt.axhline(optimal_att[1], ls='--')
if save:
    plt.savefig(f'../figs/{folder}/attack_prob_{folder}.png', 
                bbox_inches='tight', dpi=200)

In [ ]:
# #step inc is group - individual
# # compromise ~ optimality
# d = group_stat.copy()
# d['deviation_from_optimal'] = round(d['deviation_from_optimal'])
# # sns.lmplot(data = group_stat, x='deviation_from_optimal', y='step_inc', hue='risky_wpair')
# sns.lineplot(data = d, x='deviation_from_optimal', y='step_inc', style='risky_wpair', hue='predatorType')
# plt.legend().remove()

In [ ]:
# plot EV: show effet of Theta
optimal = []
lss = [':', '-', '--']
thetas = [0.7, 1, 1.4]

for t, theta in enumerate(thetas):
    for pt in [0, 1]:
        attack_prob = df_idv.query('predatorType==@pt and choice>0').groupby(['choice'], as_index=False)['currProb'].mean()
        values = (1 - attack_prob['currProb'].values) * (attack_prob['choice'] ** 2) ** theta - attack_prob['currProb'].values * (10 ** theta)
        plt.plot((values - np.min(values)) / (np.max(values) - np.min(values)), ls=lss[t], c=sns.color_palette()[pt], lw=1.8)
        plt.axvline(x=np.argmax(values),ls=lss[t], c=sns.color_palette()[pt], alpha=0.4, lw=1.2)

plt.xticks(np.arange(9), np.arange(9)+1)
plt.xlabel('step')
plt.ylabel('normalized expected reward')


legend_handles = [
    Line2D([0], [0], color='black', ls=lss[i], label=rf"$\theta$ = {thetas[i]}")
    for i in range(len(lss))
]

plt.legend(handles=legend_handles, loc='center left', fontsize=14)


In [ ]:


# # Melt to long format
# long_df = betas_df.melt(
#     id_vars=['subID', 'group'],
#     value_vars=[
#         'intercept',
#         'attack[T.True]',
#         'player_partner_diff',
#         'player_partner_diff:attack[T.True]'
#     ],
#     var_name='parameter',
#     value_name='beta'
# )


# # Set style
# # sns.set(style="whitegrid")

# # Create the FacetGrid
# g = sns.FacetGrid(
#     long_df,
#     row="parameter",
#     hue="group",
#     aspect=6,
#     height=1.5,
#     palette = compromise_palette,
#     sharex=True,
#     sharey=False
# )

# # Add KDE plots per group
# g.map(sns.kdeplot, "beta", fill=True, alpha=0.5, linewidth=1.5)

# # Add vertical lines at group means (optional)
# import numpy as np
# for ax, param in zip(g.axes.flat, long_df['parameter'].unique()):
#     for grp in long_df['group'].unique():
#         mean = long_df.query(f"parameter == '{param}' and group == '{grp}'")['beta'].mean()
#         ax.axvline(mean, linestyle='--', linewidth=1)

# g.set_titles(row_template="{row_name}", size=12)
# g.set_xlabels("Estimated beta")
# g.set_ylabels("")
# g.add_legend()
# plt.tight_layout()
# plt.show()


# #stats
# for col in ['intercept', 'attack[T.True]', 'player_partner_diff', 'player_partner_diff:attack[T.True]']:
#     group_low = betas_df.query("group == 'low'")[col]
#     group_high = betas_df.query("group == 'high'")[col]

#     # t-test
#     t_stat, p_val = ss.ttest_ind(group_low, group_high)

#     # Cohen's d
#     mean_diff = group_high.mean() - group_low.mean()
#     n1, n2 = len(group_low), len(group_high)
#     s1, s2 = group_low.std(ddof=1), group_high.std(ddof=1)
#     s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
#     d = mean_diff / s_pooled

#     print(f"{col}:\n  t = {t_stat:.2f}, p = {p_val:.4f}, Cohen's d = {d:.3f}")


In [ ]:
# #run two regressions
# model = smf.mixedlm(formula='selfBlame ~ player_partner_diff + compromise + player_partner_diff * compromise', 
#                     data=d.query('attack==True'), groups = d.query('attack==True')['subID'], 
#                     # re_formula="~ player_partner_diff + compromise + player_partner_diff * compromise"
#                     )

# result = model.fit()
# print(result.summary())


# model = smf.mixedlm(formula='selfBlame ~ player_partner_diff + compromise + player_partner_diff * compromise', 
#                     data=d.query('attack==False'), groups = d.query('attack==False')['subID'],
#                     # re_formula="~ player_partner_diff + compromise + player_partner_diff * compromise"
#                     )

# result = model.fit()
# print(result.summary())

In [ ]:


# Extract random effects (per subject)
# re = result.random_effects  # dict: subID -> series of random slopes

# # Combine with fixed effects
# fe = result.fe_params  # Series of fixed effects

# # Construct per-subject betas (fixed + random)
# sub_betas = pd.DataFrame({sub: fe.add(r) for sub, r in re.items()}).T
# sub_betas = sub_betas.reset_index().rename(columns={'index': 'subID'})

# # Melt for plotting
# df_plot = sub_betas.melt(id_vars='subID', var_name='term', value_name='beta')
# terms = ['compromise[T.low]', 'outcome[T.win]:compromise[T.low]', 'player_partner_diff:compromise[T.low]']
# # KDE plot or boxplot
# plt.figure(figsize=(8, 10))
# sns.violinplot(data=df_plot.query('term in @terms'), x='term', y='beta', inner='point', scale='width', order=terms)
# plt.xticks(rotation=45)
# plt.axhline(0, ls='--', lw=0.8, color='gray')
# plt.title("Participant-level distribution of betas (fixed + random)")
# plt.tight_layout()
# plt.show()


In [ ]:

# # Extract random effects (per subject)
# re = result.random_effects  # dict: subID -> series of random slopes
# fe = result.fe_params  # Series of fixed effects

# # Construct per-subject betas (fixed + random)
# sub_betas = pd.DataFrame({sub: fe.add(r) for sub, r in re.items()}).T
# sub_betas = sub_betas.reset_index().rename(columns={'index': 'subID'})

# # Melt for plotting
# df_plot = sub_betas.melt(id_vars='subID', var_name='term', value_name='beta')

# # Filter only selected terms
# terms = ['compromise[T.low]', 'outcome[T.win]:compromise[T.low]', 'player_partner_diff:compromise[T.low]']
# terms2 = ['Intercept', 'outcome[T.win]', 'player_partner_diff']
# df_plot1 = df_plot[df_plot['term'].isin(terms)]

# # KDE plot: one row per term (rotated layout)
# g = sns.FacetGrid(df_plot1, row='term', height=2, aspect=3, sharex=True)
# g.map(sns.kdeplot, 'beta', fill=True, alpha=0.4, linewidth=1.5)

# # df_plot2 = df_plot[df_plot['term'].isin(terms)]

# # # KDE plot: one row per term (rotated layout)
# # g = sns.FacetGrid(df_plot2, row='term', height=2, aspect=4, sharex=True)
# # g.map(sns.kdeplot, 'beta', fill=True, alpha=0.4, linewidth=1.5)

# # Add vertical lines at zero and term-wise means
# for ax, term in zip(g.axes.flatten(), terms):
#     mean_val = df_plot.query("term == @term")['beta'].mean()
#     ax.axvline(0, color='gray', linestyle='--', linewidth=0.8)
#     ax.axvline(mean_val, color='black', linestyle='-', linewidth=1.2)

# # Tidy layout
# g.set_titles(row_template="{row_name}")
# g.set_xlabels("individual betas")
# g.set_ylabels("")
# plt.tight_layout()

# if save:
#     plt.savefig(f'../figs/{folder}/blame_betas_{mymodel}_{mymodel2}_{folder}.png', 
#                 bbox_inches='tight', dpi=200)


In [ ]:
#blame ~ player_partner_diff + attack + player_partner_diff * if_attack + player_parnter_diff * if_w_high + attack * if_w_high + player_partner_diff * if_attack * if_whigh
d2 = d.copy()
d2['w_high'] = d2['compromise'] == 'high'
d2['win'] = d2['attack'].apply(lambda x: True if not x else False)
d2['win_highC'] = d2.apply(lambda x: x['win'] if x['w_high'] else 0, axis=1)
d2['player_partner_diff_highC'] = d2.apply(lambda x: x['player_partner_diff'] if x['w_high'] else 0, axis=1)
d2['int_highC'] = d2.apply(lambda x: x['player_partner_diff'] if (x['w_high'] and x['win']) else 0, axis=1)
# Run the regression using the formula
model = smf.mixedlm(formula='selfBlame ~ player_partner_diff + win + player_partner_diff * win + win_highC + player_partner_diff_highC', 
                    data=d2, groups = d['subID'])

result = model.fit(method="lbfgs", maxiter=1000)

# Print the summary of the model

print(result.summary())

In [ ]:
#distribution of player partner difference
# x = df_group.groupby(['player_partner_diff', 'subID', 'predatorType'], as_index=False)['trial'].count()
# x['player_partner_diff']= x['player_partner_diff'].astype(int)
# sns.barplot(data = x, x='player_partner_diff', y='trial', hue='predatorType')
plt.figure(figsize=(5.5, 4))
plt.hist(df_group['player_partner_diff'], bins=np.arange(-8, 9)-0.5, density=True, 
         color='black', alpha=0.7)
plt.ylabel('Frequency of trials')
plt.xlabel('$S_{player} - S_{partner}$')
if save: 
    plt.savefig(f'../figs/{folder}/ppd_dist_{folder}.png', bbox_inches='tight', dpi=200)

plt.figure(figsize=(5.5, 4))
plt.hist(df_group.query('selfBlame!=-1')['selfBlame'], bins=np.arange(0, 1.2, 0.2), density=True, 
         color='black', alpha=0.7)
plt.ylabel('Frequency of trials')
plt.xticks(np.arange(0.1, 1.1, 0.2), ['0%', '25%', '50%', '75%', '100%'])
plt.yticks(np.arange(0, 3.5), np.arange(0, 3.5)/5)
plt.xlabel('credit / blame assigned to self')
if save:
    plt.savefig(f'../figs/{folder}/blame_dist_{folder}.png', bbox_inches='tight', dpi=200)
# d= df_group.query('selfBlame!=-1').groupby(['subID', 'selfBlame'], as_index=False)['trial'].count()
# sns.boxplot(data=d, x='selfBlame', y='trial')

In [ ]:
#binarize player partner diff?
plt.figure(figsize=(4, 5))
d['player_forward'] = d['player_partner_diff'].apply(lambda x: 1 if x>0 else (-1 if x<0 else 0))
m = d.groupby(['subID', 'player_forward', 'compromise', 'attack'], 
                                    as_index=False)['selfBlame'].mean()
m = pd.merge(m.query('attack==False'), m.query('attack==True'), 
             on=['subID', 'player_forward', 'compromise'])
m['diff'] = m['selfBlame_x'] - m['selfBlame_y'] #lose - win
sns.pointplot(data = m,
               x='player_forward', y='diff', hue='compromise', dodge=0.1)

# calculate social reward

In [ ]:
social_reward = df_group.copy()
social_reward['partner_assigned_reward'] = (1 - social_reward['partnerBlame']) * social_reward['jointMoney']
social_reward['self_assigned_reward'] = social_reward['selfBlame'] * social_reward['jointMoney']
## swap blame for attack
social_reward['partnerBlame'] = social_reward.apply(lambda row: row['partnerBlame'] if row['attack']==0 else 1-row['partnerBlame'], axis=1)
social_reward['selfBlame'] = social_reward.apply(lambda row: row['selfBlame'] if row['attack']==0 else 1-row['selfBlame'], axis=1)

social_reward_g = pd.merge(social_reward.query('selfBlame>-1 and selfBlame<2').groupby(['subID', 'room'], as_index=False)[['self_assigned_reward', 'selfBlame']].mean(),
                         social_reward.query('partnerBlame>-1 and partnerBlame<2').groupby(['subID', 'room'], as_index=False)[['partner_assigned_reward', 'partnerBlame']].mean())
social_reward_g = pd.merge(social_reward_g, params_df)
social_reward_g['partnerBlame'] = 1 - social_reward_g['partnerBlame']
social_reward_g = pd.merge(social_reward_g, social_reward_g.groupby(['room'], as_index=False)['theta'].mean().rename({'theta':'theta_wpair'}, axis=1),
                         how='left')
social_reward_g['theta_wpair'] = social_reward_g.apply(lambda row: 'higher' if row['theta'] >row['theta_wpair'] else 'lower', axis=1)
social_reward_g['compromise'] = social_reward_g.apply(lambda row: 'high' if row['w']>np.median(social_reward_g['w']) else 'low', axis=1)
print(len(social_reward_g))

In [ ]:
sns.lmplot(data = social_reward_g, x='w', y='partnerBlame', 
           scatter_kws={'s':12, 'alpha':0.25, 'color':'black'}, line_kws={'color':'black'})
r,p = ss.pearsonr(social_reward_g['partnerBlame'], social_reward_g['w'])
p = f"{round(p, 3)}" if p>=0.001 else "p < 0.001"
plt.annotate(f"r = {round(r, 2)}\n{p}", xy=(0.5, 0.05), fontsize=15)
plt.ylabel('% credit attributed by partner')
plt.yticks([0, 0.2, 0.4, 0.6], ['0%', '20%', '40%', '60%'])
# sns.swarmplot(data = social_reward_g, x='compromise', y='partner_assigned_reward')
# sns.barplot(data = social_reward_g, x='compromise', y='partner_assigned_reward')
# print(ss.ttest_ind(social_reward_g.query('compromise =="high"')['partner_assigned_reward'], 
#                    social_reward_g.query('compromise =="low"')['partner_assigned_reward']))
if save:
    plt.savefig(f'../figs/{folder}/partnerBlame_w_{folder}.png', 
                bbox_inches='tight', dpi=200)

plt.figure()
sns.lmplot(data = social_reward_g, x='w', y='selfBlame',
           scatter_kws={'s':12, 'alpha':0.25, 'color':'black'}, line_kws={'color':'black'})
r, p = ss.pearsonr(social_reward_g['selfBlame'], social_reward_g['w'])
p = f"{round(p, 3)}" if p>=0.001 else "p < 0.001"
plt.annotate(f"r = {round(r, 2)}\n{p}", xy=(0.5, 0.9), fontsize=15)
plt.ylabel('% credit attributed by self')
plt.yticks([0.4, 0.6, 0.8, 1], ['40%', '60%', '80%', '100%'])
print(r, p)
if save:
    plt.savefig(f'../figs/{folder}/selfBlame_w_{folder}.png', 
                bbox_inches='tight', dpi=200)
# sns.swarmplot(data = social_reward, x='compromise', y='self_assigned_reward')
# sns.barplot(data = social_reward, x='compromise', y='self_assigned_reward')
# print(ss.ttest_ind(social_reward.query('compromise =="high"')['self_assigned_reward'], 
#                    social_reward.query('compromise =="low"')['self_assigned_reward']))

In [ ]:
sns.lmplot(data = social_reward_g
           , x='w', y='partner_assigned_reward', hue='theta_wpair')
# ss.pearsonr(social_reward['partner_assigned_reward'], social_reward['w'])

In [ ]:

# social_reward_g2 = pd.merge(social_reward.query('selfBlame>-1').groupby(['subID', 'room', 'predatorType'], as_index=False)['self_assigned_reward'].mean(),
#                          social_reward.query('partnerBlame>-1').groupby(['subID', 'room', 'predatorType'], as_index=False)['partner_assigned_reward'].mean())
social_reward_g2= pd.merge(social_reward_g, collapsed_g[['room', 'predatorType', 'cluster']])

# sns.barplot(data=social_reward_g, x='cluster', y='partner_assigned_reward')
# plt.figure()
# sns.barplot(data=social_reward_g, x='cluster', y='self_assigned_reward')

# Reshape to long format
df_long = social_reward_g2.melt(
    id_vars=['cluster'],
    value_vars=['partner_assigned_reward', 'self_assigned_reward'],
    var_name='reward_type',
    value_name='reward'
)
df_long= df_long.replace({'partner_assigned_reward':'partner assigned', 'self_assigned_reward':'self assigned'})
# Plot with hue
plt.figure(figsize=(6,5))
# myswarm(data=df_long, x='cluster', y='reward', hue='reward_type', dodge=True, legend=False)
# mybar(data=df_long, x='cluster', y='reward', hue='reward_type')
sns.boxplot(data=df_long, x='cluster', y='reward', hue='reward_type')

# plt.ylabel("Assigned reward")
plt.xlabel("")
plt.xticks(rotation=30)
plt.legend(loc='upper left')
plt.show()


In [ ]:
social_reward_g2 = pd.merge(social_reward_g2, group_stat)
social_reward_g2 = pd.merge(social_reward_g2, params_df)
mybar(data = social_reward_g2, x='cluster', y='w', hue='risky_wpair')
myswarm(data = social_reward_g2, x='cluster', y='w', hue='risky_wpair', dodge=True, legend=False)
# sns.barplot(data = social_reward_g2, x='cluster', y='w', hue='risky_wpair', palette=risk_palette)

In [ ]:
# cut group into 4 based on reward_inc and plot w: no obvious difference
d = social_reward_g2.copy()

d['reward_increased'] = d['jointMoney'] > np.median(d['jointMoney'])
sns.barplot(data = d, x='reward_increased', y='w', hue='risky_wpair', alpha=0.6)
sns.swarmplot(data = d, x='reward_increased', y='w', hue='risky_wpair', dodge=True, legend=False, alpha=0.4)

In [ ]:
# import statsmodels.formula.api as smf
social_reward_g2['step_inc_directed'] = social_reward_g2.apply(lambda row: row['step_inc'] if row['risky_wpair']=="risk-averse" else -row['step_inc'], axis=1)

# Fit OLS regression with interaction term
model = smf.ols(
    formula="partner_assigned_reward ~ risky_wpair * step_inc_directed",
    data=social_reward_g2
).fit()
# model = smf.ols(
#     formula="partnerBlame ~ risky_wpair * step_inc_directed",
#     data=social_reward_g2
# ).fit()

# Print regression results
print(model.summary())


In [ ]:
from scipy.stats import median_abs_deviation as mad

# 1) per-subject MAD of playerStep (scaled like SD under normality)
step_var = (
    df_group.groupby('subID', as_index=False)
            .agg(playerStep_var=('playerStep', lambda x: mad(x, scale='normal')))
)

# 2) merge into social_reward_g2
social_reward_g2 = social_reward_g2.merge(step_var, on='subID', how='left')

# 3) run OLS (example: predict partner_assigned_reward from the MAD)
model = smf.ols(
    formula="partnerBlame ~ step_inc + playerStep_var",
    data=social_reward_g2
).fit()

print(model.summary())


In [ ]:
test = pd.merge(group_stat.groupby(['room', 'subID'], as_index=False)[['jointMoney', 'reward_inc']].mean(),
                params_df)
print(len(test))
test['theta_wpair'] = test.sort_values(by=['room', 'theta']).groupby(['room'])['theta'].cumcount()
test = pd.merge(test.query('theta_wpair==0')[['room', 'jointMoney', 'reward_inc', 'w', 'theta']], 
                 test.query('theta_wpair==1')[['room', 'jointMoney', 'reward_inc', 'w', 'theta']],
                 on=['room'], suffixes=("_ra", "_rp"))
print(len(test))
test['avg_w'] = test['w_ra'] + test['w_rp']
test['w_diff'] = test['w_ra'] - test['w_rp'] 
test['avg_theta'] = test['theta_ra'] + test['theta_rp']
test['theta_diff'] = test['theta_ra'] - test['theta_rp']
test['reward_inc'] = (test['reward_inc_ra'] + test['reward_inc_rp']) / 2
test['jointMoney'] = (test['jointMoney_ra'] + test['jointMoney_rp']) / 2

In [ ]:
test['reward_increased'] = test['reward_inc']>0
sns.barplot(data = test, x='reward_increased', y='w_diff')

In [ ]:
# sns.barplot(data=social_reward_g, x='risky_wpair', y='partner_assigned_reward', dodge=True, legend=False)
# sns.barplot(data=social_reward_g, x='cluster', y='reward_inc', hue='risky_wpair')

# is there any correlation between lr and risky / risk-averse and w

sns.scatterplot(data = test, x='w_ra', y='w_rp')
plt.plot([-1, 1], [-1, 1])

plt.figure()
sns.scatterplot(data = test, x='theta_ra', y='theta_rp')
plt.plot([0, 1.5], [0, 1.5])


# plt.figure()
# test['ra_compromised_more'] = test['w_ra'] > test['w_rp']
# sns.boxplot(data = test, x='ra_compromised_more', y='jointMoney')
# ss.ttest_ind(test.query('ra_compromised_more==True')['jointMoney'], test.query('ra_compromised_more==False')['jointMoney'])

# plt.figure()
# sns.boxplot(data = test, x='ra_compromised_more', y='reward_inc')
# ss.ttest_ind(test.query('ra_compromised_more==True')['reward_inc'], test.query('ra_compromised_more==False')['reward_inc'])

In [ ]:
# group_stat.sort_values(by='reward_inc', ascending=False)['room'].values

In [ ]:
# visualize one dyad's time series 
room = 103
plot_d = df_group.query('room==@room and predatorType==1 and playerID==1')
plt.plot(plot_d['trial'], plot_d['playerStep'], '^')
plt.plot(plot_d['trial'], plot_d['partnerStep'], 'o')
# plt.plot(data = df_group.query('room==@room and predatorType==0'),x='trial', y='partnerStep')

In [ ]:
# # test1: does consistency predicts joint performance? - nope
# def mad(x):
#     return np.median(np.abs(x - np.median(x)))

# Step 1: aggregate MAD and mean
d = (
    df_group.query('playerID==0')
    .groupby(['subID', 'predatorType'], as_index=False)
    .agg({
        'playerStep': mad,
        'partnerStep': mad,
        'jointMoney': 'mean'
    })
)

# Step 2: assign smaller vs larger MAD
d['stepVar_low'] = d[['playerStep', 'partnerStep']].min(axis=1)
d['stepVar_high'] = d[['playerStep', 'partnerStep']].max(axis=1)

# Optional: drop originals if you only want the labeled vars
# d = d.drop(columns=['playerStep', 'partnerStep'])

smf.ols('jointMoney ~ stepVar_low * stepVar_high', data = d).fit().summary()

In [ ]:
# Collapse to dyad-trial level
df_dyad = (
    df_group.groupby(['room','trial','predatorType'], as_index=False)
    .agg({
        'playerStep': list,
        'partnerStep': list,
        'jointMoney': 'mean'
    })
)

# compute symmetric predictors
df_dyad['avg_step'] = df_dyad.apply(lambda r: np.mean([np.mean(r['playerStep']),
                                                       np.mean(r['partnerStep'])]), axis=1)

df_dyad['step_diff'] = df_dyad.apply(lambda r: abs(np.mean(r['playerStep']) - 
                                                   np.mean(r['partnerStep'])), axis=1)

# If you want trial-to-trial variability per partner, first collapse player-level stats:
df_stats = (
    df_group.groupby(['room','predatorType','playerID'], as_index=False)
    .agg(step_mean=('playerStep','mean'),
         step_var=('playerStep','var'))
)

# then recombine into dyad-level averages and differences
df_sym = (
    df_stats.groupby(['room','predatorType'], as_index=False)
    .agg(avg_step=('step_mean','mean'),
         step_diff=('step_mean', lambda x: abs(x.iloc[0]-x.iloc[1])),
         avg_var=('step_var','mean'),
         var_diff=('step_var', lambda x: abs(x.iloc[0]-x.iloc[1])))
)

# merge jointMoney back in
df_sym = df_sym.merge(
    df_group.groupby(['room','predatorType'], as_index=False)['jointMoney'].mean(),
    on=['room','predatorType']
)


model = smf.ols(
    formula="jointMoney ~ avg_step + step_diff + avg_var + var_diff",
    data=df_sym
).fit()

print(model.summary())


In [ ]:
# does w correlate with accurate prediction of partner's choice?
df_group['abs_pred_partner_diff'] = df_group['prediction'] - df_group['partnerStep']
d = df_group.query('prediction>0').groupby(['subID'], as_index=False)['abs_pred_partner_diff'].mean()
d = pd.merge(d, params_df)
d['abs_w'] = np.abs(d['w'])
sns.regplot(data = d, x='abs_w', y='abs_pred_partner_diff')
print(ss.pearsonr(d['abs_w'], d['abs_pred_partner_diff']))

# plt.figure()
# group_stat['step_inc_directed']=group_stat.apply(lambda row: row['step_inc'] if row['risky_wpair']=="risk-averse" else -row['step_inc'], axis=1)
# d = pd.merge(group_stat.groupby('subID', as_index=False)['step_inc_directed'].apply(lambda x: np.mean(x)), d)
# sns.regplot(data = d, x='step_inc_directed', y='abs_pred_partner_diff')
# print(ss.pearsonr(d['step_inc_directed'], d['abs_pred_partner_diff']))
d = group_stat.copy()
d = pd.merge(df_group.groupby(['subID'], as_index=False)['partnerStep'].apply(mad).rename({'partnerStep':'partnerStep_var'}, axis=1), d)
d['partnerStep_var_mc'] = d['partnerStep_var'] - np.mean(d['partnerStep_var'])
smf.ols('step_inc ~ self_partner_risk_diff + partnerStep_var_mc', data=d).fit().summary()

In [ ]:


# For each room, find min and max reward, and the jointMoney (assuming it's the same across both subIDs)
room_level = (
    group_stat.groupby(['room', 'predatorType'], as_index=False)
              .agg(
                  lower_reward=('reward', 'min'),
                  higher_reward=('reward', 'max'),
                  mean_reward = ('reward', 'mean'),
                  lower_risk = ('individual', 'min'),
                  higher_risk = ('individual', 'max'),
                  mean_risk = ('individual', 'mean'),
                  jointMoney=('jointMoney', 'mean')  # or 'first' if identical
              )
)


# model = smf.ols(
#     formula="jointMoney ~ lower_reward + higher_reward",
#     data=room_level
# ).fit()

# model = smf.ols(
#     formula="jointMoney ~ lower_risk + higher_risk",
#     data=room_level
# ).fit()


print(model.summary())


In [ ]:
# model = smf.ols(formula='reward_inc ~ avg_theta * w_ra + theta_rp * w_rp', 
#                     data=test).fit()
# model = smf.ols(formula='reward_inc ~ avg_theta + avg_w + w_diff + theta_diff', 
#                     data=test).fit()
model = smf.ols(formula='reward_inc ~ theta_diff * avg_w', 
                    data=test).fit()

print(model.summary())

# new: can you predict reward from bias?


In [ ]:
d = group_stat.groupby(['room'], as_index=False)[['finalStep', 'jointMoney', 'step_inc', 'individual', 'reward_inc']].mean()

In [ ]:
ego_bias = df_group.query('selfBlame>0').groupby(['subID', 'attack'], as_index=False)['selfBlame'].mean()
ego_bias = pd.merge(ego_bias.query('attack==True'), ego_bias.query('attack==False'), on=['subID'], suffixes=['_lose', '_win'])
ego_bias['ego_bias'] = ego_bias['selfBlame_win'] - ego_bias['selfBlame_lose']

In [ ]:
d = group_stat.copy()
d['step_inc_directed'] = d.apply(lambda row: row['step_inc'] if row['risky_wpair']=="risk-averse" else -row['step_inc'], axis=1)
d = pd.merge(d, ego_bias).groupby(['room'], as_index=False)[['finalStep', 'jointMoney', 'step_inc_directed', 'ego_bias', 'reward_inc']].mean()

In [ ]:
##Run the regression using the formula

model = smf.mixedlm(formula='jointMoney ~ finalStep + step_inc_directed', 
                    data=d, groups = d['room'])

result = model.fit(method="lbfgs", maxiter=1000)

# Print the summary of the model
print(result.summary())

print(ss.pearsonr(d['finalStep'], d['step_inc_directed']))

In [ ]:
##Run the regression using the formula
d = group_stat.copy()
d['step_inc_directed'] = d.apply(lambda row: row['step_inc'] if row['risky_wpair']=="risk-averse" else -row['step_inc'], axis=1)

model = smf.mixedlm(
    formula='reward_inc ~ C(risky_wpair) * step_inc_directed * C(predatorType)', 
    data=d, 
    groups=d['subID']
)
result = model.fit()
print(result.summary())


# print(ss.pearsonr(d['finalStep'], d['step_inc_directed']))

In [ ]:
# d = pd.merge(df_group.query('playerID==0 and selfBlame>0')[['trial', 'room', 'predatorType', 'jointMoney', 'finalStep', 'selfBlame']], 
#              df_group.query('playerID==1 and selfBlame>0')[['trial', 'room', 'predatorType', 'jointMoney', 'finalStep', 'selfBlame']],
#              on=['trial', 'room', 'predatorType', 'jointMoney', 'finalStep'], how='inner')
# d['blame_async'] = np.abs(1 - d['selfBlame_x'] - d['selfBlame_y'])

# # d = pd.merge(group_stat, df_group.query('selfBlame>0').groupby(['subID'], as_index=False)['selfBlame'].mean())
# d = d.groupby(['room'], as_index=False)[['finalStep', 'jointMoney', 'blame_async']].mean()

# model = smf.mixedlm(formula='jointMoney ~ blame_async', 
#                     data=d, groups = d['room'])

# result = model.fit(method="lbfgs", maxiter=1000)

# # Print the summary of the model
# print(result.summary())

# print(ss.pearsonr(d['finalStep'], d['step_inc_directed']))

# test correlations

In [ ]:

# Step 1: Create a pivot table: rows = trials, columns = players, values = choice
pivoted = df_group.pivot_table(index=['room', 'trial', 'predatorType'],
                         columns='playerID',
                         values='playerStep')

# Step 2: For each room, compute the correlation matrix across players
room_correlations = {}

for room_id, group in pivoted.groupby(level=['room', 'predatorType']):
    # Drop trials with missing data
    trial_data = group.droplevel(['room', 'predatorType']).dropna()
    
    # Compute correlation between players
    corr_matrix = trial_data.corr()
    
    # Optionally store or flatten the values
    room_correlations[room_id] = corr_matrix

# To see correlation for room 'A':
# print(room_correlations['A'])

# Optionally flatten the upper triangle for summary
summary = {
    room: corr.where(~np.tril(np.ones(corr.shape)).astype(bool)).stack().mean()
    for room, corr in room_correlations.items()
}

summary = pd.DataFrame.from_dict(summary, orient='index', columns=['mean_pairwise_correlation'])
summary['room'] = [i[0] for i in summary.index]
summary['predatorType'] = [i[1] for i in summary.index]
summary = pd.merge(summary, collapsed_g[['room', 'predatorType', 'cluster', 'idvDiff']])
summary = summary.dropna()

In [ ]:


# Plot
sns.lmplot(data=summary, x='idvDiff', y='mean_pairwise_correlation')
print(ss.pearsonr(summary['idvDiff'], summary['mean_pairwise_correlation']))

# Assume your DataFrame looks like this:
# summary_df has columns: 'cluster', 'mean_pairwise_correlation'

# Group by cluster and test each against 0
results = []

for cluster, group in summary.dropna().groupby('cluster'):
    values = group['mean_pairwise_correlation']
    t_stat, p_value = ss.ttest_1samp(values, popmean=0)
    
    results.append({
        'cluster': cluster,
        'n': len(values),
        'mean_corr': values.mean(),
        't_stat': t_stat,
        'p_value': p_value
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)


In [ ]:
results_df

## KG: testing if we can say that bias in blame only matters when outcome does not align with position. 

In [ ]:
##new column based on player partner diff: if player partner diff is positive, then player is more risky than partner, otherwise less risky

test['player_partner_diff_binary'] = np.sign(test['player_partner_diff'])  # Convert to binary

##if attack is True, then attack_binary is -1, otherwise 1
test['attack_binary'] = test['attack'].apply(lambda x: -1 if x else 1) 
##multiply attack and player partner diff binary to get a new column
test['interaction'] = test['attack_binary'] * test['player_partner_diff_binary']
# test = test.dropna()
##turn interaction in int
# test['interaction'] = test['interaction'].astype(int)

test.head()

In [ ]:
##t-test the interaction effect on selfBlame
r, p = ss.ttest_ind(test.query('interaction==1')['selfBlame'], 
                    test.query('interaction==-1')['selfBlame'])

print(f"t = {r}, p = {p}")

##test effect of compromise on selfBlame -- ttest
##for interaction == -1 
r, p = ss.ttest_ind(
    test[(test['interaction'] == -1) & (test['compromised'])]['selfBlame'],
    test[(test['interaction'] == -1) & (~test['compromised'])]['selfBlame']
)

print(f"t = {r}, p = {p}")

##for interaction == 1
r, p = ss.ttest_ind(
    test[(test['interaction'] == 1) & (test['compromised'])]['selfBlame'],
    test[(test['interaction'] == 1) & (~test['compromised'])]['selfBlame']
)
print(f"t = {r}, p = {p}")



In [ ]:
sns.lineplot(data=test.groupby(['subID', 'player_partner_diff', 'interaction', 'compromised'], as_index=False)['selfBlame'].mean(),
              x='player_partner_diff', y='selfBlame', hue='compromised',
            style='interaction', palette='mako', 
)

In [ ]:
sns.catplot(data=test[test.interaction != 0], x='interaction', y='selfBlame', hue='compromised',
            kind='box', palette='mako', 
)

In [ ]:
sns.catplot(data=test[test.interaction != 0], x='interaction', y='selfBlame2', hue='compromised',
            kind='box', palette='mako', 
)

In [ ]:
# df_cleaned = df_group.sort_values(by=['room', 'trial'])
# df_cleaned['partner_changed'] = df_cleaned['partnerStep'] - df_cleaned.groupby(['sub', 'block'])['partnerStep'].shift()
# df_cleaned['player_changed'] = df_cleaned['playerStep'] - df_cleaned.groupby(['sub', 'block'])['playerStep'].shift()
# df_cleaned['player_changed_binned'] = pd.cut(df_cleaned['player_changed'], bins = [-8, -2, 1, 9])
# df_cleaned['partner_changed_binned'] = pd.cut(df_cleaned['partner_changed'], bins = [-8, -2, 1, 9])
# d = df_cleaned.query('step_rt<8 and playerStep>0 and partnerStep>0 and selfBlame!=-1'
#                      ).groupby(['player_changed_binned', 'partner_changed_binned'], as_index=False
#                    )['selfBlame'].mean()
# d = d.pivot(columns='player_changed_binned', 
#                         index='partner_changed_binned', values='selfBlame')
# sns.heatmap(d, vmin=0, vmax=1, cmap='coolwarm')


# for the new design only

In [ ]:
group_stat_filtered

In [ ]:
# Data processing for individual and group foraging
#get  post individual choice
gi2 = df_idv2.groupby(['subID', 'predatorType'], as_index=False)[['choice', 'reward']].mean()
gi2 = gi2.rename({'choice':'choice_post', 'reward':'reward_post'}, axis=1)

# get group choice
g_new = pd.merge(gi2, group_stat_filtered[['room', 'risky_wpair', 'predatorType', 'subID', 'group', 'jointMoney', 'individual', 'reward']])


In [ ]:
# #compare money increase between risky and risk-averse individuals
g_new['reward_inc'] = g_new['reward_post'] - g_new['jointMoney'] / 2
g_new['step_inc'] = g_new['choice_post'] - g_new['group']

plt.figure(figsize=(4, 5))
sns.violinplot(data=g_new, y='reward_inc', x='risky_wpair', palette=risk_palette)
plt.ylabel('reward increase after dyad')

plt.figure(figsize=(4, 5))
sns.violinplot(data=g_new, y='step_inc', x='risky_wpair', palette=risk_palette, 
            # flierprops=dict(markersize=8, alpha=0.7)
            )
plt.xlabel('riskiness within dyad')
plt.ylabel('foraging step increase after dyad')
plt.text(1, 8, "n.s.", ha='center', va='center', fontsize=16) #this is based on regression runned later
plt.text(0, 8, "**", ha='center', va='center', fontsize=18) #this is based on regression runned later
plt.ylim([-9, 9])
plt.axhline(0, color='black', lw=0.75, ls='--')

plt.savefig(f'../figs/{folder}/step_inc_{folder}.png', 
            bbox_inches='tight', dpi=200)


print(ss.ttest_1samp(g_new.query('risky_wpair=="risk-prone"')['step_inc'], 0))
sample =g_new.query('risky_wpair=="risk-prone"')['step_inc']
mean_diff = 0 - np.mean(sample)  # or just np.mean(sample)
std_dev = np.std(sample, ddof=1)  # use ddof=1 for sample std
cohens_d = mean_diff / std_dev
print(f"Cohen's d for risk-prone: {cohens_d}")


print(ss.ttest_1samp(g_new.query('risky_wpair=="risk-averse"')['step_inc'], 0))
sample =g_new.query('risky_wpair=="risk-averse"')['step_inc']
mean_diff = np.mean(sample) - 0  # or just np.mean(sample)
std_dev = np.std(sample, ddof=1)  # use ddof=1 for sample std
cohens_d = mean_diff / std_dev
print(f"Cohen's d for risk-averse: {cohens_d}")




In [ ]:



ego_bias = get_ego_bias(df_group, groupby_cols=['subID', 'predatorType'])
# plt.hist(ego_bias['ego_bias'], bins=np.arange(-1, 1.2, 0.2), color='black', alpha=0.7)
# g_new = pd.merge(g_new, ego_bias[['subID', 'ego_bias']], on='subID', how='left')
sns.lmplot(data=g_new, y='step_inc', x='ego_bias', hue='risky_wpair', palette=risk_palette, 
            # flierprops=dict(markersize=8, alpha=0.7)
            )

model = smf.ols(formula='step_inc ~ ego_bias * individual', data=g_new).fit()
print(model.summary())

In [ ]:
# #compare money increase between risky and risk-averse individuals
g_new['step_inc_prepost'] = g_new['choice_post'] - g_new['individual']
# g_new['step_inc'] = g_new['choice_post'] - g_new['group']
g_new['reward_inc_prepost'] =g_new['reward_post'] -  g_new['reward']

plt.figure(figsize=(4, 5))
sns.violinplot(data=g_new, y='reward_inc_prepost', x='risky_wpair', palette=risk_palette)


plt.figure(figsize=(4, 5))
sns.violinplot(data=g_new, y='step_inc_prepost', x='risky_wpair', palette=risk_palette, 
            # flierprops=dict(markersize=8, alpha=0.7)
            )

# need to correct for multiple comparison?
plt.xlabel('riskiness within dyad')
plt.ylabel('individual foraging step increase \n(post - pre dyad)')
plt.text(0, 7, "n.s.", ha='center', va='center', fontsize=16) #this is based on regression runned later
plt.text(1, 7, "****", ha='center', va='center', fontsize=18) #this is based on regression runned later
plt.ylim([-6, 8])
plt.axhline(0, color='black', lw=0.75, ls='--')



print(ss.ttest_1samp(g_new.query('risky_wpair=="risk-prone"')['step_inc_prepost'], 0))
sample =g_new.query('risky_wpair=="risk-prone"')['step_inc_prepost']
mean_diff = 0 - np.mean(sample)  # or just np.mean(sample)
std_dev = np.std(sample, ddof=1)  # use ddof=1 for sample std
cohens_d = mean_diff / std_dev
print(f"Cohen's d for risk-prone: {cohens_d}")


print(ss.ttest_1samp(g_new.query('risky_wpair=="risk-averse"')['step_inc_prepost'], 0))
sample =g_new.query('risky_wpair=="risk-averse"')['step_inc_prepost']
mean_diff = np.mean(sample) - 0  # or just np.mean(sample)
std_dev = np.std(sample, ddof=1)  # use ddof=1 for sample std
cohens_d = mean_diff / std_dev
print(f"Cohen's d for risk-averse: {cohens_d}")


plt.savefig(f'../figs/{folder}/step_inc_prepost_{folder}.png', 
            bbox_inches='tight', dpi=200)


In [ ]:

# ----------------------------
# Helper: weighted linear regression
# ----------------------------
def wls(X, y, w):
    # Solve (X^T W X) beta = X^T W y
    WX = X * w[:, None]
    beta = np.linalg.solve(WX.T @ X, WX.T @ y)
    return beta  # [intercept, slope]

# ----------------------------
# Mixture of 2 lines via EM
# ----------------------------
def mixture_two_lines(x, y, max_iter=500, tol=1e-6, n_restarts=5, seed=0):
    """
    Fits y ~ a_k + b_k * x + Normal(0, sigma_k^2), k in {0,1}
    with mixing weights pi (for k=1) and 1-pi (for k=0).
    Returns best run by log-likelihood.
    """
    rng = np.random.default_rng(seed)
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()
    n = len(y)
    X = np.c_[np.ones(n), x]  # intercept + x

    best = None

    for r in range(n_restarts):
        # --- initialization ---
        # split by x median to get two crude lines (you can try k-means on y, etc.)
        mask = (x > np.median(x)) if r % 2 == 0 else (y > np.median(y))
        # OLS on each half to get starting betas
        beta0 = np.linalg.lstsq(X[~mask], y[~mask], rcond=None)[0]
        beta1 = np.linalg.lstsq(X[mask],  y[mask],  rcond=None)[0]
        # noise stds
        resid0 = y[~mask] - X[~mask] @ beta0
        resid1 = y[mask]  - X[mask]  @ beta1
        sigma0 = np.std(resid0) + 1e-6
        sigma1 = np.std(resid1) + 1e-6
        pi = 0.5  # mixture weight P(component=1)

        prev_ll = -np.inf

        for it in range(max_iter):
            # --- E-step: responsibilities ---
            # log-likelihoods per component
            mu0 = X @ beta0
            mu1 = X @ beta1
            # Gaussian log pdfs
            c0 = -0.5*np.log(2*np.pi*sigma0**2)
            c1 = -0.5*np.log(2*np.pi*sigma1**2)
            ll0 = c0 - 0.5*((y - mu0)**2) / (sigma0**2) + np.log(1 - pi + 1e-12)
            ll1 = c1 - 0.5*((y - mu1)**2) / (sigma1**2) + np.log(pi + 1e-12)

            # normalize to get gamma = P(k=1 | x,y)
            m = np.maximum(ll0, ll1)  # for stability
            denom = np.exp(ll0 - m) + np.exp(ll1 - m)
            gamma1 = np.exp(ll1 - m) / denom  # resp for comp 1
            gamma0 = 1.0 - gamma1

            # total log-likelihood
            ll = np.sum(m + np.log(denom + 1e-12))

            # --- M-step: update params with weights ---
            # update betas via weighted least squares
            beta0 = wls(X, y, gamma0)
            beta1 = wls(X, y, gamma1)

            # update sigmas
            mu0 = X @ beta0
            mu1 = X @ beta1
            sigma0 = np.sqrt((gamma0 * (y - mu0)**2).sum() / (gamma0.sum() + 1e-12)) + 1e-12
            sigma1 = np.sqrt((gamma1 * (y - mu1)**2).sum() / (gamma1.sum() + 1e-12)) + 1e-12

            # update mixing weight
            pi = gamma1.mean()

            # check convergence
            if np.abs(ll - prev_ll) < tol * (1.0 + np.abs(prev_ll)):
                break
            prev_ll = ll

        # store run
        out = dict(
            beta0=beta0, beta1=beta1,
            sigma0=sigma0, sigma1=sigma1,
            pi=pi, loglike=prev_ll,
            gamma1=gamma1
        )
        if best is None or out["loglike"] > best["loglike"]:
            best = out

    # compute simple ICs
    # parameters: 2 betas per line (4), 2 sigmas, 1 mixing weight = 7
    k = 7
    n = len(y)
    aic = 2*k - 2*best["loglike"]
    bic = k*np.log(n) - 2*best["loglike"]
    best.update(dict(aic=aic, bic=bic))
    return best

# ----------------------------
# Example usage
# ----------------------------
if __name__ == "__main__":
    rng = np.random.default_rng(123)

    # Generate synthetic data from two lines
    # n0, n1 = 150, 150
    # x0 = rng.uniform(-2, 2, n0); y0 =  1.0 + 0.5*x0 + rng.normal(0, 0.25, n0)
    # x1 = rng.uniform(-2, 2, n1); y1 = -0.5 + 1.5*x1 + rng.normal(0, 0.25, n1)

    # x = np.concatenate([x0, x1])
    # y = np.concatenate([y0, y1])
    x = collapsed_g['grp_idv_step_diff_risk_prone']
    y = collapsed_g['grp_idv_step_diff_risk_averse']
    n = len(collapsed_g)

    fit = mixture_two_lines(x, y, n_restarts=10, seed=1)
    print("LogLik:", fit["loglike"])
    print("AIC:", fit["aic"], "BIC:", fit["bic"])
    print("pi (comp1):", fit["pi"])
    print("beta0 (comp0): intercept=%.3f slope=%.3f" % tuple(fit["beta0"]))
    print("beta1 (comp1): intercept=%.3f slope=%.3f" % tuple(fit["beta1"]))

    # Soft cluster assignments (posterior P(comp1|x,y))
    gamma1 = fit["gamma1"]
    # Hard labels if you need them:
    labels = (gamma1 >= 0.5).astype(int)


In [ ]:

from scipy.stats import chi2

# --- 1-line baseline (MLE + Gaussian loglik) ---
def fit_one_line(x, y, return_params=False):
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()
    n = len(y)
    X = np.c_[np.ones(n), x]

    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta
    sigma = np.sqrt(np.mean(resid**2) + 1e-12)

    ll = -0.5*n*np.log(2*np.pi*sigma**2) - 0.5*np.sum(resid**2)/sigma**2

    if return_params:
        return {"intercept": beta[0], "slope": beta[1], "sigma": sigma}
    else:
        k = 3  # intercept, slope, sigma
        return ll, k

# --- Bootstrap LRT: H0=one line vs H1=two-line mixture ---
def bootstrap_lrt_one_vs_two(x, y, fit_two_lines_fn, B=300, seed=123, n_restarts=10):
    """
    Parametric bootstrap LRT for mixture vs single line.
    - x, y: data arrays
    - fit_two_lines_fn: your mixture_two_lines function
      must return dict with 'loglike' and (optionally) 'aic','bic'
    - B: number of bootstrap replicates
    Returns dict with LR_obs, p_value, LR_boot (array), and fitted model stats.
    """
    rng = np.random.default_rng(seed)
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()

    # Fit models to observed data
    ll1, k1 = fit_one_line(x, y)                   # H0 fit
    fit2 = fit_two_lines_fn(x, y, n_restarts=n_restarts, seed=seed)  # H1 fit
    ll2 = fit2["loglike"]
    LR_obs = 2.0 * (ll2 - ll1)

    # Get H0 parameters for simulation
    theta0 = fit_one_line(x, y, return_params=True)

    # Bootstrap under H0
    LR_boot = np.empty(B)
    for b in range(B):
        yb = theta0["intercept"] + theta0["slope"] * x + rng.normal(0, theta0["sigma"], size=len(x))
        ll1_b, _ = fit_one_line(x, yb)
        fit2_b = fit_two_lines_fn(x, yb, n_restarts=n_restarts, seed=seed + b + 1)
        LR_boot[b] = 2.0 * (fit2_b["loglike"] - ll1_b)

    # p-value = proportion of bootstrap LRs >= observed LR
    p = (1 + np.sum(LR_boot >= LR_obs)) / (B + 1)

    # Convenience ICs for reporting
    n = len(y)
    k2 = 7  # two lines: 2 betas per line (4) + 2 sigmas + 1 mixing weight
    aic1 = 2*k1 - 2*ll1
    bic1 = k1*np.log(n) - 2*ll1
    aic2 = 2*k2 - 2*ll2
    bic2 = k2*np.log(n) - 2*ll2

    return {
        "LR_obs": LR_obs,
        "p_value": p,
        "LR_boot": LR_boot,
        "one_line": {"ll": ll1, "k": k1, "AIC": aic1, "BIC": bic1},
        "two_lines": {"ll": ll2, "k": k2, "AIC": aic2, "BIC": bic2, **{k: v for k, v in fit2.items() if k not in ["loglike"]}},
    }

# --- Example with your data ---
if __name__ == "__main__":
    # Your x, y:
    x = collapsed_g['grp_idv_step_diff_risk_prone'].to_numpy()
    y = collapsed_g['grp_idv_step_diff_risk_averse'].to_numpy()

    res = bootstrap_lrt_one_vs_two(x, y, mixture_two_lines, B=300, seed=42, n_restarts=10)

    print(f"Observed LR = {res['LR_obs']:.3f}")
    print(f"Bootstrap p-value = {res['p_value']:.4f}")
    print("1-line:  AIC = %.2f  BIC = %.2f" % (res["one_line"]["AIC"], res["one_line"]["BIC"]))
    print("2-lines: AIC = %.2f  BIC = %.2f" % (res["two_lines"]["AIC"], res["two_lines"]["BIC"]))
    print("Mixing weight (pi) ~", res["two_lines"].get("pi", None))
    print("beta0 (comp0):", res["two_lines"].get("beta0", None))
    print("beta1 (comp1):", res["two_lines"].get("beta1", None))


In [ ]:

def lr_stat_one_vs_two(x, y, fit_two_lines, fit_one_line):
    # fit_one_line returns (ll, k); fit_two_lines returns dict with 'loglike' and k=7
    ll1, k1 = fit_one_line(x, y)         # e.g., OLS with Gaussian noise
    fit2 = fit_two_lines(x, y)           # your EM mixture
    ll2, k2 = fit2['loglike'], 7
    LR = 2*(ll2 - ll1)
    return LR, (ll1, k1), (ll2, k2)

def bootstrap_lrt(x, y, fit_two_lines, fit_one_line, B=200, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    LR_obs, (ll1, k1), (ll2, k2) = lr_stat_one_vs_two(x, y, fit_two_lines, fit_one_line)
    # simulate under H0 using the fitted 1-line parameters
    theta0 = fit_one_line(x, y, return_params=True)  # get beta, sigma
    LR_boot = []
    for _ in range(B):
        yb = theta0['intercept'] + theta0['slope']*x + rng.normal(0, theta0['sigma'], size=len(x))
        LR_b, *_ = lr_stat_one_vs_two(x, yb, fit_two_lines, fit_one_line)
        LR_boot.append(LR_b)
    p = (1 + np.sum(np.array(LR_boot) >= LR_obs)) / (B + 1)
    return {"LR_obs": LR_obs, "p": p, "LR_boot": np.array(LR_boot)}


In [ ]:
# Fit 1-line model
ll1, k1 = fit_one_line(x, y)
aic1 = 2*k1 - 2*ll1
bic1 = k1*np.log(n) - 2*ll1
print("1-line model: AIC=%.2f BIC=%.2f" % (aic1, bic1))

# Fit 2-line model
fit2 = mixture_two_lines(x, y, n_restarts=10, seed=1)
print("2-line model: AIC=%.2f BIC=%.2f" % (fit2["aic"], fit2["bic"]))


In [ ]:
# do a cluster of subjects on compromise, egobias, reciprocity, risk...
